In [1]:
# =============================================================================
# CELL 1 — NOTEBOOK RUNTIME PARAMETERS
# QuickBooks Online Incremental Bronze Ingestion
#
# Fabric configuration:
# Mark only this cell as the notebook parameter cell.
#
# Purpose:
# These values provide safe development defaults and are overridden by the
# Fabric pipeline at runtime.
#
# Security:
# Do not place access tokens, refresh tokens, client secrets, realm IDs,
# connection strings, or other credentials in this parameter cell.
# =============================================================================


# -----------------------------------------------------------------------------
# Pipeline execution context
# -----------------------------------------------------------------------------

# Supplied by the Fabric pipeline using @pipeline().RunId.
# A blank value is permitted for controlled manual notebook testing.
p_pipeline_run_id = ""


# -----------------------------------------------------------------------------
# Source-system context
# -----------------------------------------------------------------------------

# Source application identifier.
p_source_system = "QBO"

# Company or legal-entity identifier.
p_source_company = "ES01"

# Source API environment: SANDBOX or PRODUCTION.
p_source_environment = "SANDBOX"


# -----------------------------------------------------------------------------
# Metadata-driven entity selection
# -----------------------------------------------------------------------------

# JSON array supplied by the pipeline Lookup activity.
# Each item describes one source entity and its ingestion configuration.
# A blank value allows the notebook to use its controlled development fallback.
p_entities_json = ""


# -----------------------------------------------------------------------------
# Incremental-load configuration
# -----------------------------------------------------------------------------

# Supported execution mode for this notebook.
p_load_type = "INCREMENTAL"

# Reprocess a small period before the stored watermark to protect against
# timing boundaries, delayed source commits, and timestamp precision issues.
p_watermark_overlap_minutes = 5


# -----------------------------------------------------------------------------
# Failure-handling policy
# -----------------------------------------------------------------------------

# True:
# Fail the notebook when any entity fails, preventing partial financial loads
# from being reported as successful.
#
# False:
# Continue processing remaining entities and record individual failures.
p_fail_on_entity_error = True

StatementMeta(, ac938067-ae54-42e4-8eda-d118e4fa985e, 3, Finished, Available, Finished, False)

In [2]:
# =============================================================================
# CELL 2 — PRIVATE QUICKBOOKS CONFIGURATION
# QuickBooks Online Incremental Bronze Ingestion
#
# IMPORTANT:
# =============================================================================


# -----------------------------------------------------------------------------
# QuickBooks OAuth development credentials
# -----------------------------------------------------------------------------

QBO_CLIENT_ID = "xxxxxxxxxxxxx"

QBO_CLIENT_SECRET = "xxxxxxxxx"

QBO_REFRESH_TOKEN = "xxxxxxxxxx"
# QuickBooks company identifier.
QBO_REALM_ID = "xxxxxxxx"


# -----------------------------------------------------------------------------
# Environment and API configuration
# -----------------------------------------------------------------------------

QBO_ENVIRONMENT = str(p_source_environment).strip().upper()

QBO_API_BASE_URLS = {
    "SANDBOX": "https://sandbox-quickbooks.api.intuit.com",
    "PRODUCTION": "https://quickbooks.api.intuit.com",
}

if QBO_ENVIRONMENT not in QBO_API_BASE_URLS:
    raise ValueError(
        "p_source_environment must be either SANDBOX or PRODUCTION."
    )

QBO_API_BASE_URL = QBO_API_BASE_URLS[QBO_ENVIRONMENT]

QBO_TOKEN_URL = (
    "https://oauth.platform.intuit.com/oauth2/v1/tokens/bearer"
)

QBO_MINOR_VERSION = "75"


# -----------------------------------------------------------------------------
# Validate required credentials
# -----------------------------------------------------------------------------

required_credentials = {
    "QBO_CLIENT_ID": QBO_CLIENT_ID,
    "QBO_CLIENT_SECRET": QBO_CLIENT_SECRET,
    "QBO_REFRESH_TOKEN": QBO_REFRESH_TOKEN,
    "QBO_REALM_ID": QBO_REALM_ID,
}

missing_credentials = [
    name
    for name, value in required_credentials.items()
    if value is None
    or not str(value).strip()
    or str(value).strip().startswith("<")
]

if missing_credentials:
    raise ValueError(
        "Replace the following QuickBooks configuration placeholders: "
        + ", ".join(missing_credentials)
    )


# -----------------------------------------------------------------------------
# Validate non-secret configuration
# -----------------------------------------------------------------------------

if not str(QBO_REALM_ID).strip().isdigit():
    raise ValueError(
        "QBO_REALM_ID must contain only numeric characters."
    )

if not str(QBO_MINOR_VERSION).strip().isdigit():
    raise ValueError(
        "QBO_MINOR_VERSION must contain only numeric characters."
    )

expected_api_hosts = {
    "SANDBOX": "sandbox-quickbooks.api.intuit.com",
    "PRODUCTION": "quickbooks.api.intuit.com",
}

expected_api_host = expected_api_hosts[QBO_ENVIRONMENT]

if expected_api_host not in QBO_API_BASE_URL:
    raise ValueError(
        f"QBO_API_BASE_URL does not match the requested "
        f"{QBO_ENVIRONMENT} environment."
    )


# -----------------------------------------------------------------------------
# Safe operational logging
# -----------------------------------------------------------------------------

realm_id_text = str(QBO_REALM_ID).strip()

masked_realm_id = (
    "*" * max(len(realm_id_text) - 4, 0)
    + realm_id_text[-4:]
)

print("=" * 80)
print("QUICKBOOKS PRIVATE CONFIGURATION")
print("=" * 80)
print(f"Source system       : {p_source_system}")
print(f"Source company      : {p_source_company}")
print(f"Environment         : {QBO_ENVIRONMENT}")
print(f"API base URL        : {QBO_API_BASE_URL}")
print(f"Realm ID            : {masked_realm_id}")
print(f"Minor version       : {QBO_MINOR_VERSION}")
print("Credential values were not printed.")
print("Configuration validation: SUCCEEDED")
print("=" * 80)

StatementMeta(, ac938067-ae54-42e4-8eda-d118e4fa985e, 4, Finished, Available, Finished, False)

QUICKBOOKS PRIVATE CONFIGURATION
Source system       : QBO
Source company      : ES01
Environment         : SANDBOX
API base URL        : https://sandbox-quickbooks.api.intuit.com
Realm ID            : ************4915
Minor version       : 75
Credential values were not printed.
Configuration validation: SUCCEEDED


In [7]:
# =============================================================================
# CELL 3 — RUNTIME INITIALISATION AND QUICKBOOKS AUTHENTICATION
# QuickBooks Online Incremental Bronze Ingestion
#
# Purpose:
# 1. Resolve the Fabric pipeline or manual execution context.
# 2. Normalise and validate runtime parameters.
# 3. Validate the QuickBooks configuration loaded by Cell 2.
# 4. Exchange the refresh token for a short-lived access token.
# 5. Construct the standard HTTP headers used by later ingestion cells.
#
# Prerequisites:
# - Cell 1 must define the notebook parameters.
# - Cell 2 must define the QuickBooks configuration variables.
#
# Security:
# Never print the access token, client secret or refresh token.
# =============================================================================


# -----------------------------------------------------------------------------
# Imports
# -----------------------------------------------------------------------------

import time
import uuid
from datetime import datetime, timedelta, timezone
from typing import Any, Dict, Optional

import requests
from requests.auth import HTTPBasicAuth


# -----------------------------------------------------------------------------
# Constants
# -----------------------------------------------------------------------------

SUPPORTED_ENVIRONMENTS = {
    "SANDBOX",
    "PRODUCTION",
}

SUPPORTED_LOAD_TYPES = {
    "INCREMENTAL",
}

TRANSIENT_HTTP_STATUS_CODES = {
    408,  # Request timeout
    425,  # Too early
    429,  # Too many requests
    500,  # Internal server error
    502,  # Bad gateway
    503,  # Service unavailable
    504,  # Gateway timeout
}

NON_RETRYABLE_AUTH_STATUS_CODES = {
    400,
    401,
    403,
}

DEFAULT_TOKEN_EXPIRY_SECONDS = 3600
TOKEN_EXPIRY_SAFETY_MINUTES = 5
MAXIMUM_TOKEN_REQUEST_ATTEMPTS = 3
INITIAL_RETRY_DELAY_SECONDS = 5
MAXIMUM_RETRY_DELAY_SECONDS = 60
REQUEST_TIMEOUT_SECONDS = 60


# =============================================================================
# 1. GENERAL HELPER FUNCTIONS
# =============================================================================

def utc_now() -> datetime:
    """Return the current timezone-aware UTC timestamp."""
    return datetime.now(timezone.utc)


def normalise_string(
    value: Any,
    default: str = "",
) -> str:
    """Convert a runtime value to a trimmed string."""
    if value is None:
        return default

    normalised_value = str(value).strip()

    if not normalised_value:
        return default

    return normalised_value


def normalise_boolean(
    value: Any,
    default: bool = False,
) -> bool:
    """
    Convert common pipeline and notebook Boolean representations
    into a Python Boolean.
    """
    if value is None:
        return default

    if isinstance(value, bool):
        return value

    normalised_value = str(value).strip().lower()

    true_values = {
        "true",
        "1",
        "yes",
        "y",
    }

    false_values = {
        "false",
        "0",
        "no",
        "n",
    }

    if normalised_value in true_values:
        return True

    if normalised_value in false_values:
        return False

    raise ValueError(
        "Cannot convert the supplied value to Boolean. "
        f"Received: {value!r}"
    )


def normalise_integer(
    value: Any,
    parameter_name: str,
) -> int:
    """Convert a notebook or pipeline parameter to an integer."""
    try:
        return int(value)

    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"{parameter_name} must be a valid integer. "
            f"Received: {value!r}"
        ) from exc


def safe_json_response(
    response: requests.Response,
) -> Dict[str, Any]:
    """
    Return the HTTP response as JSON when possible.

    If the response is not JSON, return a limited text representation
    without exposing request headers or authentication credentials.
    """
    try:
        response_payload = response.json()

        if isinstance(response_payload, dict):
            return response_payload

        return {
            "response_payload": response_payload,
        }

    except ValueError:
        return {
            "raw_response": normalise_string(
                response.text,
            )[:2000]
        }


def get_runtime_context_value(
    runtime_context: Any,
    key: str,
    default: Any = None,
) -> Any:
    """
    Read a value from the Fabric runtime context safely.

    Fabric runtime context may behave like a dictionary or expose
    attributes depending on the execution environment.
    """
    if runtime_context is None:
        return default

    if isinstance(runtime_context, dict):
        return runtime_context.get(
            key,
            default,
        )

    try:
        return runtime_context.get(
            key,
            default,
        )
    except (AttributeError, TypeError):
        return getattr(
            runtime_context,
            key,
            default,
        )


def calculate_retry_delay(
    attempt_number: int,
    retry_after_header: Optional[str] = None,
) -> int:
    """
    Calculate the retry delay.

    Honour a numeric Retry-After response header when supplied.
    Otherwise, use capped exponential backoff.
    """
    if (
        retry_after_header
        and str(retry_after_header).strip().isdigit()
    ):
        return min(
            int(str(retry_after_header).strip()),
            MAXIMUM_RETRY_DELAY_SECONDS,
        )

    exponential_delay = (
        INITIAL_RETRY_DELAY_SECONDS
        * (2 ** (attempt_number - 1))
    )

    return min(
        exponential_delay,
        MAXIMUM_RETRY_DELAY_SECONDS,
    )


# =============================================================================
# 2. RESOLVE FABRIC EXECUTION CONTEXT
# =============================================================================

try:
    RUNTIME_CONTEXT = notebookutils.runtime.context
except Exception:
    # A missing context is acceptable during controlled manual testing.
    RUNTIME_CONTEXT = {}


pipeline_context_flags = [
    get_runtime_context_value(
        RUNTIME_CONTEXT,
        "isForPipeline",
        False,
    ),
    get_runtime_context_value(
        RUNTIME_CONTEXT,
        "isPipelineRun",
        False,
    ),
]

IS_PIPELINE_RUN = any(
    normalise_boolean(
        context_flag,
        False,
    )
    for context_flag in pipeline_context_flags
)


RECEIVED_PIPELINE_RUN_ID = normalise_string(
    p_pipeline_run_id
)

if RECEIVED_PIPELINE_RUN_ID:
    PIPELINE_RUN_ID = RECEIVED_PIPELINE_RUN_ID
    RUN_ID_SOURCE = "PIPELINE_PARAMETER"

else:
    # This fallback supports controlled manual notebook testing.
    PIPELINE_RUN_ID = str(
        uuid.uuid4()
    )
    RUN_ID_SOURCE = "MANUAL_TEST_UUID"


RUN_STARTED_UTC = utc_now()


# =============================================================================
# 3. NORMALISE RUNTIME PARAMETERS
# =============================================================================

SOURCE_SYSTEM = normalise_string(
    p_source_system,
    "QBO",
).upper()

SOURCE_COMPANY = normalise_string(
    p_source_company,
    "ES01",
).upper()

SOURCE_ENVIRONMENT = normalise_string(
    p_source_environment,
    "SANDBOX",
).upper()

LOAD_TYPE = normalise_string(
    p_load_type,
    "INCREMENTAL",
).upper()

WATERMARK_OVERLAP_MINUTES = normalise_integer(
    p_watermark_overlap_minutes,
    "p_watermark_overlap_minutes",
)

FAIL_ON_ENTITY_ERROR = normalise_boolean(
    p_fail_on_entity_error,
    True,
)


# =============================================================================
# 4. VALIDATE RUNTIME PARAMETERS
# =============================================================================

if SOURCE_SYSTEM != "QBO":
    raise ValueError(
        "This notebook supports only the QBO source system. "
        f"Received: {SOURCE_SYSTEM!r}"
    )


if not SOURCE_COMPANY:
    raise ValueError(
        "p_source_company cannot be blank."
    )


if SOURCE_ENVIRONMENT not in SUPPORTED_ENVIRONMENTS:
    raise ValueError(
        "p_source_environment must be SANDBOX or PRODUCTION. "
        f"Received: {SOURCE_ENVIRONMENT!r}"
    )


if LOAD_TYPE not in SUPPORTED_LOAD_TYPES:
    raise ValueError(
        "This notebook currently supports only INCREMENTAL loading. "
        f"Received: {LOAD_TYPE!r}"
    )


if WATERMARK_OVERLAP_MINUTES < 0:
    raise ValueError(
        "p_watermark_overlap_minutes cannot be negative."
    )


if WATERMARK_OVERLAP_MINUTES > 60:
    raise ValueError(
        "p_watermark_overlap_minutes cannot exceed 60 minutes."
    )


if IS_PIPELINE_RUN and not RECEIVED_PIPELINE_RUN_ID:
    raise RuntimeError(
        "This notebook appears to be running from a Fabric pipeline, "
        "but p_pipeline_run_id is blank. Configure the notebook "
        "activity parameter with @pipeline().RunId."
    )


# =============================================================================
# 5. VALIDATE QUICKBOOKS CONFIGURATION FROM CELL 2
# =============================================================================

required_qbo_configuration = {
    "QBO_CLIENT_ID": QBO_CLIENT_ID,
    "QBO_CLIENT_SECRET": QBO_CLIENT_SECRET,
    "QBO_REFRESH_TOKEN": QBO_REFRESH_TOKEN,
    "QBO_REALM_ID": QBO_REALM_ID,
    "QBO_API_BASE_URL": QBO_API_BASE_URL,
    "QBO_TOKEN_URL": QBO_TOKEN_URL,
    "QBO_MINOR_VERSION": QBO_MINOR_VERSION,
}


invalid_configuration = [
    configuration_name
    for configuration_name, configuration_value
    in required_qbo_configuration.items()
    if configuration_value is None
    or not str(configuration_value).strip()
    or str(configuration_value).strip().startswith("<")
]


if invalid_configuration:
    raise ValueError(
        "The following QuickBooks configuration values are missing "
        "or still contain placeholders: "
        + ", ".join(invalid_configuration)
    )


QBO_CLIENT_ID = str(
    QBO_CLIENT_ID
).strip()

QBO_CLIENT_SECRET = str(
    QBO_CLIENT_SECRET
).strip()

QBO_REFRESH_TOKEN = str(
    QBO_REFRESH_TOKEN
).strip()

QBO_REALM_ID = str(
    QBO_REALM_ID
).strip()

QBO_API_BASE_URL = str(
    QBO_API_BASE_URL
).strip().rstrip("/")

QBO_TOKEN_URL = str(
    QBO_TOKEN_URL
).strip()

QBO_MINOR_VERSION = str(
    QBO_MINOR_VERSION
).strip()


if not QBO_REALM_ID.isdigit():
    raise ValueError(
        "QBO_REALM_ID must contain only numeric characters."
    )


if not QBO_MINOR_VERSION.isdigit():
    raise ValueError(
        "QBO_MINOR_VERSION must contain only numeric characters."
    )


expected_api_hosts = {
    "SANDBOX": "sandbox-quickbooks.api.intuit.com",
    "PRODUCTION": "quickbooks.api.intuit.com",
}

expected_api_host = expected_api_hosts[
    SOURCE_ENVIRONMENT
]


if expected_api_host not in QBO_API_BASE_URL:
    raise ValueError(
        "The QuickBooks API URL does not match the requested "
        f"{SOURCE_ENVIRONMENT} environment. "
        f"Expected host: {expected_api_host!r}."
    )


if (
    SOURCE_ENVIRONMENT == "PRODUCTION"
    and "sandbox-" in QBO_API_BASE_URL.lower()
):
    raise ValueError(
        "The environment is PRODUCTION, but the configured "
        "QuickBooks API URL points to the sandbox endpoint."
    )


# =============================================================================
# 6. QUICKBOOKS ACCESS-TOKEN FUNCTION
# =============================================================================

def request_qbo_access_token(
    maximum_attempts: int = MAXIMUM_TOKEN_REQUEST_ATTEMPTS,
) -> Dict[str, Any]:
    """
    Exchange the configured QuickBooks refresh token for a short-lived
    access token.

    Retry policy:
    - Retry network exceptions and transient HTTP responses.
    - Stop immediately for invalid credentials, invalid refresh tokens
      and other non-retryable HTTP responses.
    """
    if maximum_attempts < 1:
        raise ValueError(
            "maximum_attempts must be at least 1."
        )

    request_headers = {
        "Accept": "application/json",
        "Content-Type": "application/x-www-form-urlencoded",
    }

    request_payload = {
        "grant_type": "refresh_token",
        "refresh_token": QBO_REFRESH_TOKEN,
    }

    for attempt_number in range(
        1,
        maximum_attempts + 1,
    ):
        try:
            response = requests.post(
                QBO_TOKEN_URL,
                headers=request_headers,
                data=request_payload,
                auth=HTTPBasicAuth(
                    QBO_CLIENT_ID,
                    QBO_CLIENT_SECRET,
                ),
                timeout=REQUEST_TIMEOUT_SECONDS,
            )

        except requests.RequestException as exc:
            if attempt_number >= maximum_attempts:
                raise RuntimeError(
                    "The QuickBooks token request failed because of "
                    "a network or connection error after "
                    f"{maximum_attempts} attempts."
                ) from exc

            delay_seconds = calculate_retry_delay(
                attempt_number=attempt_number,
            )

            print(
                "QuickBooks token request encountered a network error. "
                f"Attempt {attempt_number} of {maximum_attempts}. "
                f"Retrying in {delay_seconds} seconds."
            )

            time.sleep(
                delay_seconds
            )
            continue


        response_details = safe_json_response(
            response
        )


        if response.status_code == 200:
            access_token = normalise_string(
                response_details.get(
                    "access_token"
                )
            )

            if not access_token:
                raise RuntimeError(
                    "QuickBooks returned HTTP 200 but did not return "
                    "an access token."
                )

            returned_refresh_token = normalise_string(
                response_details.get(
                    "refresh_token"
                ),
                QBO_REFRESH_TOKEN,
            )

            try:
                expires_in_seconds = int(
                    response_details.get(
                        "expires_in",
                        DEFAULT_TOKEN_EXPIRY_SECONDS,
                    )
                )
            except (TypeError, ValueError) as exc:
                raise RuntimeError(
                    "QuickBooks returned an invalid expires_in value."
                ) from exc

            if expires_in_seconds <= 0:
                raise RuntimeError(
                    "QuickBooks returned a non-positive access-token "
                    "expiry period."
                )

            return {
                "access_token": access_token,
                "refresh_token": returned_refresh_token,
                "expires_in": expires_in_seconds,
                "token_received_utc": utc_now(),
                "http_status": response.status_code,
                "attempt_number": attempt_number,
            }


        if (
            response.status_code
            in NON_RETRYABLE_AUTH_STATUS_CODES
        ):
            raise RuntimeError(
                "QuickBooks authentication was rejected. "
                f"HTTP status: {response.status_code}. "
                f"Response details: {response_details}"
            )


        if (
            response.status_code
            not in TRANSIENT_HTTP_STATUS_CODES
        ):
            raise RuntimeError(
                "QuickBooks returned a non-retryable token-response error. "
                f"HTTP status: {response.status_code}. "
                f"Response details: {response_details}"
            )


        if attempt_number >= maximum_attempts:
            raise RuntimeError(
                "The QuickBooks token request failed after "
                f"{maximum_attempts} attempts. "
                f"Final HTTP status: {response.status_code}. "
                f"Response details: {response_details}"
            )


        delay_seconds = calculate_retry_delay(
            attempt_number=attempt_number,
            retry_after_header=response.headers.get(
                "Retry-After"
            ),
        )

        print(
            "QuickBooks token service returned a transient response. "
            f"HTTP status: {response.status_code}. "
            f"Attempt {attempt_number} of {maximum_attempts}. "
            f"Retrying in {delay_seconds} seconds."
        )

        time.sleep(
            delay_seconds
        )


    raise RuntimeError(
        "QuickBooks token processing ended unexpectedly."
    )


# =============================================================================
# 7. REQUEST AND RESOLVE THE ACCESS TOKEN
# =============================================================================

token_result = request_qbo_access_token()


ACCESS_TOKEN = token_result[
    "access_token"
]

LATEST_REFRESH_TOKEN = token_result[
    "refresh_token"
]

TOKEN_RECEIVED_UTC = token_result[
    "token_received_utc"
]

ACCESS_TOKEN_EXPIRES_UTC = (
    TOKEN_RECEIVED_UTC
    + timedelta(
        seconds=token_result["expires_in"]
    )
)

ACCESS_TOKEN_REFRESH_BY_UTC = (
    ACCESS_TOKEN_EXPIRES_UTC
    - timedelta(
        minutes=TOKEN_EXPIRY_SAFETY_MINUTES
    )
)

REFRESH_TOKEN_ROTATED = (
    LATEST_REFRESH_TOKEN
    != QBO_REFRESH_TOKEN
)


# =============================================================================
# 8. CREATE STANDARD QUICKBOOKS REQUEST HEADERS
# =============================================================================

QBO_REQUEST_HEADERS = {
    "Authorization": (
        f"Bearer {ACCESS_TOKEN}"
    ),
    "Accept": "application/json",
    "Content-Type": "application/json",
}


# =============================================================================
# 9. SAFE RUNTIME SUMMARY
# =============================================================================

masked_realm_id = (
    "*" * max(
        len(QBO_REALM_ID) - 4,
        0,
    )
    + QBO_REALM_ID[-4:]
)


print("=" * 80)
print("QBO INCREMENTAL RUNTIME INITIALISATION")
print("=" * 80)
print(f"Pipeline run ID              : {PIPELINE_RUN_ID}")
print(f"Run ID source                : {RUN_ID_SOURCE}")
print(f"Pipeline execution detected  : {IS_PIPELINE_RUN}")
print(f"Run started UTC              : {RUN_STARTED_UTC.isoformat()}")
print(f"Source system                : {SOURCE_SYSTEM}")
print(f"Source company               : {SOURCE_COMPANY}")
print(f"Environment                  : {SOURCE_ENVIRONMENT}")
print(f"Realm ID                     : {masked_realm_id}")
print(f"Load type                    : {LOAD_TYPE}")
print(
    "Watermark overlap minutes  : "
    f"{WATERMARK_OVERLAP_MINUTES}"
)
print(
    "Fail on entity error        : "
    f"{FAIL_ON_ENTITY_ERROR}"
)
print(
    "Authentication attempts     : "
    f"{token_result['attempt_number']}"
)
print(
    "Access-token expiry UTC     : "
    f"{ACCESS_TOKEN_EXPIRES_UTC.isoformat()}"
)
print(
    "Access-token refresh-by UTC : "
    f"{ACCESS_TOKEN_REFRESH_BY_UTC.isoformat()}"
)
print(
    "Replacement refresh token   : "
    f"{REFRESH_TOKEN_ROTATED}"
)
print("QuickBooks authentication    : SUCCEEDED")
print("=" * 80)


if REFRESH_TOKEN_ROTATED:
    print(
        "WARNING: QuickBooks returned a replacement refresh token. "
        "The current development configuration does not persist rotated "
        "refresh tokens automatically. Update the private credential "
        "configuration securely before a later run if required."
    )


# Never print:
# - ACCESS_TOKEN
# - QBO_CLIENT_SECRET
# - QBO_REFRESH_TOKEN
# - LATEST_REFRESH_TOKEN

StatementMeta(, 981929cc-545c-427d-8156-72bdf95a7ed5, 9, Finished, Available, Finished, False)

QBO INCREMENTAL RUNTIME INITIALISATION
Pipeline run ID              : 722b23a8-b13b-46a5-af73-c023f36263bf
Run ID source                : MANUAL_TEST_UUID
Pipeline execution detected  : False
Run started UTC              : 2026-08-14T15:58:33.191982+00:00
Source system                : QBO
Source company               : ES01
Environment                  : SANDBOX
Realm ID                     : ************4915
Load type                    : INCREMENTAL
Watermark overlap minutes  : 5
Fail on entity error        : True
Authentication attempts     : 1
Access-token expiry UTC     : 2026-08-14T16:58:34.022232+00:00
Access-token refresh-by UTC : 2026-08-14T16:53:34.022232+00:00
Replacement refresh token   : False
QuickBooks authentication    : SUCCEEDED


In [24]:
# =============================================================================
# CELL 4 — RESOLVE AND VALIDATE INCREMENTAL ENTITY METADATA
# QuickBooks Online Incremental Bronze Ingestion
#
# Purpose:
# 1. Read entity metadata supplied by the Fabric Lookup activity.
# 2. Provide a controlled fallback for manual notebook testing.
# 3. Normalise and validate every metadata record.
# 4. Publish ordered runtime metadata for downstream cells.
#
# Production standard:
# ctl.ingestion_config remains the production source of truth.
# The manual fallback is only for controlled notebook development.
# =============================================================================


# -----------------------------------------------------------------------------
# Imports
# -----------------------------------------------------------------------------

import json
import re
from collections import Counter
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

from pyspark.sql.types import (
    IntegerType,
    StringType,
    StructField,
    StructType,
)


# -----------------------------------------------------------------------------
# Supported metadata values
# -----------------------------------------------------------------------------

SUPPORTED_QBO_SOURCE_OBJECTS = {
    "Account",
    "Customer",
    "Vendor",
    "Item",
    "Class",
    "Department",
    "Term",
    "Invoice",
    "Payment",
    "CreditMemo",
    "Bill",
    "BillPayment",
    "Purchase",
    "PurchaseOrder",
    "JournalEntry",
}

SUPPORTED_SOURCE_TYPES = {
    "REST_API",
}

SUPPORTED_ENTITY_LOAD_TYPES = {
    "INCREMENTAL",
}

SUPPORTED_WATERMARK_COLUMNS = {
    "MetaData.LastUpdatedTime",
}

VALID_IDENTIFIER_PATTERN = re.compile(
    r"^[A-Za-z_][A-Za-z0-9_]*$"
)


# =============================================================================
# 1. MANUAL-DEVELOPMENT METADATA
# =============================================================================
#
# Used only when:
# - the notebook is run manually; and
# - p_entities_json is blank.
#
# During Fabric pipeline execution, the Lookup activity must supply metadata
# through p_entities_json.
# =============================================================================

MANUAL_TEST_ENTITIES = [
    {
        "ingestion_id": 2001,
        "source_system": "QBO",
        "source_company": "ES01",
        "source_object": "Account",
        "source_type": "REST_API",
        "target_schema": "dbo",
        "target_table": "qbo_es_accounts",
        "target_line_table": None,
        "load_type": "INCREMENTAL",
        "watermark_column": "MetaData.LastUpdatedTime",
        "last_watermark": None,
        "api_query_template": (
            "SELECT * FROM Account "
            "STARTPOSITION {start_position} "
            "MAXRESULTS {page_size}"
        ),
        "response_collection": "Account",
        "page_size": 1000,
        "supports_pagination": 1,
        "load_sequence": 10,
        "is_active": 1,
    },
    {
        "ingestion_id": 2002,
        "source_system": "QBO",
        "source_company": "ES01",
        "source_object": "Customer",
        "source_type": "REST_API",
        "target_schema": "dbo",
        "target_table": "qbo_es_customers",
        "target_line_table": None,
        "load_type": "INCREMENTAL",
        "watermark_column": "MetaData.LastUpdatedTime",
        "last_watermark": None,
        "api_query_template": (
            "SELECT * FROM Customer "
            "STARTPOSITION {start_position} "
            "MAXRESULTS {page_size}"
        ),
        "response_collection": "Customer",
        "page_size": 1000,
        "supports_pagination": 1,
        "load_sequence": 20,
        "is_active": 1,
    },
    {
        "ingestion_id": 2003,
        "source_system": "QBO",
        "source_company": "ES01",
        "source_object": "Vendor",
        "source_type": "REST_API",
        "target_schema": "dbo",
        "target_table": "qbo_es_vendors",
        "target_line_table": None,
        "load_type": "INCREMENTAL",
        "watermark_column": "MetaData.LastUpdatedTime",
        "last_watermark": None,
        "api_query_template": (
            "SELECT * FROM Vendor "
            "STARTPOSITION {start_position} "
            "MAXRESULTS {page_size}"
        ),
        "response_collection": "Vendor",
        "page_size": 1000,
        "supports_pagination": 1,
        "load_sequence": 30,
        "is_active": 1,
    },
    {
        "ingestion_id": 2004,
        "source_system": "QBO",
        "source_company": "ES01",
        "source_object": "Item",
        "source_type": "REST_API",
        "target_schema": "dbo",
        "target_table": "qbo_es_items",
        "target_line_table": None,
        "load_type": "INCREMENTAL",
        "watermark_column": "MetaData.LastUpdatedTime",
        "last_watermark": None,
        "api_query_template": (
            "SELECT * FROM Item "
            "STARTPOSITION {start_position} "
            "MAXRESULTS {page_size}"
        ),
        "response_collection": "Item",
        "page_size": 1000,
        "supports_pagination": 1,
        "load_sequence": 40,
        "is_active": 1,
    },
    {
        "ingestion_id": 2005,
        "source_system": "QBO",
        "source_company": "ES01",
        "source_object": "Class",
        "source_type": "REST_API",
        "target_schema": "dbo",
        "target_table": "qbo_es_classes",
        "target_line_table": None,
        "load_type": "INCREMENTAL",
        "watermark_column": "MetaData.LastUpdatedTime",
        "last_watermark": None,
        "api_query_template": (
            "SELECT * FROM Class "
            "STARTPOSITION {start_position} "
            "MAXRESULTS {page_size}"
        ),
        "response_collection": "Class",
        "page_size": 1000,
        "supports_pagination": 1,
        "load_sequence": 50,
        "is_active": 1,
    },
    {
        "ingestion_id": 2006,
        "source_system": "QBO",
        "source_company": "ES01",
        "source_object": "Department",
        "source_type": "REST_API",
        "target_schema": "dbo",
        "target_table": "qbo_es_departments",
        "target_line_table": None,
        "load_type": "INCREMENTAL",
        "watermark_column": "MetaData.LastUpdatedTime",
        "last_watermark": None,
        "api_query_template": (
            "SELECT * FROM Department "
            "STARTPOSITION {start_position} "
            "MAXRESULTS {page_size}"
        ),
        "response_collection": "Department",
        "page_size": 1000,
        "supports_pagination": 1,
        "load_sequence": 60,
        "is_active": 1,
    },
    {
        "ingestion_id": 2007,
        "source_system": "QBO",
        "source_company": "ES01",
        "source_object": "Term",
        "source_type": "REST_API",
        "target_schema": "dbo",
        "target_table": "qbo_es_terms",
        "target_line_table": None,
        "load_type": "INCREMENTAL",
        "watermark_column": "MetaData.LastUpdatedTime",
        "last_watermark": None,
        "api_query_template": (
            "SELECT * FROM Term "
            "STARTPOSITION {start_position} "
            "MAXRESULTS {page_size}"
        ),
        "response_collection": "Term",
        "page_size": 1000,
        "supports_pagination": 1,
        "load_sequence": 70,
        "is_active": 1,
    },
    {
        "ingestion_id": 2010,
        "source_system": "QBO",
        "source_company": "ES01",
        "source_object": "Invoice",
        "source_type": "REST_API",
        "target_schema": "dbo",
        "target_table": "qbo_es_invoices",
        "target_line_table": "qbo_es_invoice_lines",
        "load_type": "INCREMENTAL",
        "watermark_column": "MetaData.LastUpdatedTime",
        "last_watermark": None,
        "api_query_template": (
            "SELECT * FROM Invoice "
            "STARTPOSITION {start_position} "
            "MAXRESULTS {page_size}"
        ),
        "response_collection": "Invoice",
        "page_size": 1000,
        "supports_pagination": 1,
        "load_sequence": 100,
        "is_active": 1,
    },
    {
        "ingestion_id": 2011,
        "source_system": "QBO",
        "source_company": "ES01",
        "source_object": "Payment",
        "source_type": "REST_API",
        "target_schema": "dbo",
        "target_table": "qbo_es_payments",
        "target_line_table": "qbo_es_payment_lines",
        "load_type": "INCREMENTAL",
        "watermark_column": "MetaData.LastUpdatedTime",
        "last_watermark": None,
        "api_query_template": (
            "SELECT * FROM Payment "
            "STARTPOSITION {start_position} "
            "MAXRESULTS {page_size}"
        ),
        "response_collection": "Payment",
        "page_size": 1000,
        "supports_pagination": 1,
        "load_sequence": 110,
        "is_active": 1,
    },
    {
        "ingestion_id": 2012,
        "source_system": "QBO",
        "source_company": "ES01",
        "source_object": "CreditMemo",
        "source_type": "REST_API",
        "target_schema": "dbo",
        "target_table": "qbo_es_credit_memos",
        "target_line_table": "qbo_es_credit_memo_lines",
        "load_type": "INCREMENTAL",
        "watermark_column": "MetaData.LastUpdatedTime",
        "last_watermark": None,
        "api_query_template": (
            "SELECT * FROM CreditMemo "
            "STARTPOSITION {start_position} "
            "MAXRESULTS {page_size}"
        ),
        "response_collection": "CreditMemo",
        "page_size": 1000,
        "supports_pagination": 1,
        "load_sequence": 120,
        "is_active": 1,
    },
    {
        "ingestion_id": 2013,
        "source_system": "QBO",
        "source_company": "ES01",
        "source_object": "Bill",
        "source_type": "REST_API",
        "target_schema": "dbo",
        "target_table": "qbo_es_bills",
        "target_line_table": "qbo_es_bill_lines",
        "load_type": "INCREMENTAL",
        "watermark_column": "MetaData.LastUpdatedTime",
        "last_watermark": None,
        "api_query_template": (
            "SELECT * FROM Bill "
            "STARTPOSITION {start_position} "
            "MAXRESULTS {page_size}"
        ),
        "response_collection": "Bill",
        "page_size": 1000,
        "supports_pagination": 1,
        "load_sequence": 130,
        "is_active": 1,
    },
    {
        "ingestion_id": 2014,
        "source_system": "QBO",
        "source_company": "ES01",
        "source_object": "BillPayment",
        "source_type": "REST_API",
        "target_schema": "dbo",
        "target_table": "qbo_es_bill_payments",
        "target_line_table": None,
        "load_type": "INCREMENTAL",
        "watermark_column": "MetaData.LastUpdatedTime",
        "last_watermark": None,
        "api_query_template": (
            "SELECT * FROM BillPayment "
            "STARTPOSITION {start_position} "
            "MAXRESULTS {page_size}"
        ),
        "response_collection": "BillPayment",
        "page_size": 1000,
        "supports_pagination": 1,
        "load_sequence": 140,
        "is_active": 1,
    },
    {
        "ingestion_id": 2015,
        "source_system": "QBO",
        "source_company": "ES01",
        "source_object": "Purchase",
        "source_type": "REST_API",
        "target_schema": "dbo",
        "target_table": "qbo_es_purchases",
        "target_line_table": "qbo_es_purchase_lines",
        "load_type": "INCREMENTAL",
        "watermark_column": "MetaData.LastUpdatedTime",
        "last_watermark": None,
        "api_query_template": (
            "SELECT * FROM Purchase "
            "STARTPOSITION {start_position} "
            "MAXRESULTS {page_size}"
        ),
        "response_collection": "Purchase",
        "page_size": 1000,
        "supports_pagination": 1,
        "load_sequence": 150,
        "is_active": 1,
    },
    {
        "ingestion_id": 2016,
        "source_system": "QBO",
        "source_company": "ES01",
        "source_object": "PurchaseOrder",
        "source_type": "REST_API",
        "target_schema": "dbo",
        "target_table": "qbo_es_purchase_orders",
        "target_line_table": "qbo_es_purchase_order_lines",
        "load_type": "INCREMENTAL",
        "watermark_column": "MetaData.LastUpdatedTime",
        "last_watermark": None,
        "api_query_template": (
            "SELECT * FROM PurchaseOrder "
            "STARTPOSITION {start_position} "
            "MAXRESULTS {page_size}"
        ),
        "response_collection": "PurchaseOrder",
        "page_size": 1000,
        "supports_pagination": 1,
        "load_sequence": 160,
        "is_active": 1,
    },
    {
        "ingestion_id": 2017,
        "source_system": "QBO",
        "source_company": "ES01",
        "source_object": "JournalEntry",
        "source_type": "REST_API",
        "target_schema": "dbo",
        "target_table": "qbo_es_journal_entries",
        "target_line_table": "qbo_es_journal_entry_lines",
        "load_type": "INCREMENTAL",
        "watermark_column": "MetaData.LastUpdatedTime",
        "last_watermark": None,
        "api_query_template": (
            "SELECT * FROM JournalEntry "
            "STARTPOSITION {start_position} "
            "MAXRESULTS {page_size}"
        ),
        "response_collection": "JournalEntry",
        "page_size": 1000,
        "supports_pagination": 1,
        "load_sequence": 170,
        "is_active": 1,
    },
]


# =============================================================================
# 2. HELPER FUNCTIONS
# =============================================================================

def parse_optional_datetime(
    value: Any,
) -> Optional[datetime]:
    """Parse an optional metadata datetime without inventing a value."""
    if value is None:
        return None

    value_text = str(value).strip()

    if not value_text or value_text.upper() == "NULL":
        return None

    normalised_value = value_text.replace(
        "Z",
        "+00:00",
    )

    try:
        parsed_value = datetime.fromisoformat(
            normalised_value
        )

        if parsed_value.tzinfo is None:
            parsed_value = parsed_value.replace(
                tzinfo=timezone.utc
            )

        return parsed_value.astimezone(
            timezone.utc
        )

    except ValueError:
        pass

    supported_formats = [
        "%Y-%m-%d %H:%M:%S.%f",
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%d",
    ]

    for datetime_format in supported_formats:
        try:
            parsed_value = datetime.strptime(
                value_text,
                datetime_format,
            )

            return parsed_value.replace(
                tzinfo=timezone.utc
            )

        except ValueError:
            continue

    raise ValueError(
        f"Unable to parse last_watermark value: {value!r}"
    )


def normalise_nullable_string(
    value: Any,
) -> Optional[str]:
    """Return a trimmed string or None."""
    if value is None:
        return None

    value_text = str(value).strip()

    if not value_text or value_text.upper() == "NULL":
        return None

    return value_text


def normalise_metadata_integer(
    value: Any,
    default: int,
    field_name: str,
) -> int:
    """Convert metadata values safely to integers."""
    if value is None or str(value).strip() == "":
        return default

    try:
        return int(value)

    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"{field_name} must be a valid integer. "
            f"Received: {value!r}"
        ) from exc


def parse_metadata_payload(
    payload_text: str,
) -> List[Dict[str, Any]]:
    """
    Parse metadata passed by a Fabric Lookup activity.

    Supported forms:
    1. JSON list
    2. {"value": [...]}
    3. {"firstRow": {...}}
    """
    try:
        parsed_payload = json.loads(
            payload_text
        )

    except json.JSONDecodeError as exc:
        raise ValueError(
            "p_entities_json does not contain valid JSON."
        ) from exc

    if isinstance(parsed_payload, list):
        return parsed_payload

    if isinstance(parsed_payload, dict):
        lookup_values = parsed_payload.get(
            "value"
        )

        if isinstance(lookup_values, list):
            return lookup_values

        first_row = parsed_payload.get(
            "firstRow"
        )

        if isinstance(first_row, dict):
            return [first_row]

    raise ValueError(
        "p_entities_json must contain a JSON list, a Fabric "
        "Lookup 'value' array, or a 'firstRow' object."
    )


def validate_identifier(
    identifier_value: Optional[str],
    field_name: str,
    ingestion_id: int,
    allow_none: bool = False,
) -> Optional[str]:
    """Validate a schema or table name used by downstream Spark logic."""
    if identifier_value is None:
        if allow_none:
            return None

        raise ValueError(
            f"Ingestion ID {ingestion_id} has a null {field_name}."
        )

    identifier_text = str(
        identifier_value
    ).strip()

    if not identifier_text:
        if allow_none:
            return None

        raise ValueError(
            f"Ingestion ID {ingestion_id} has a blank {field_name}."
        )

    if not VALID_IDENTIFIER_PATTERN.fullmatch(
        identifier_text
    ):
        raise ValueError(
            f"Ingestion ID {ingestion_id} has an invalid "
            f"{field_name}: {identifier_text!r}."
        )

    return identifier_text


def find_duplicates(
    values: List[Any],
) -> List[Any]:
    """Return sorted values that occur more than once."""
    counts = Counter(values)

    return sorted(
        value
        for value, count in counts.items()
        if count > 1
    )


# =============================================================================
# 3. RESOLVE THE METADATA SOURCE
# =============================================================================

received_entities_json = (
    str(p_entities_json).strip()
    if p_entities_json is not None
    else ""
)

if received_entities_json:
    raw_entity_metadata = parse_metadata_payload(
        received_entities_json
    )

    METADATA_SOURCE = "FABRIC_LOOKUP_ACTIVITY"

else:
    if IS_PIPELINE_RUN:
        raise RuntimeError(
            "The notebook is running from a Fabric pipeline, "
            "but p_entities_json is blank. Configure the notebook "
            "activity to receive the Lookup output."
        )

    raw_entity_metadata = MANUAL_TEST_ENTITIES

    METADATA_SOURCE = "MANUAL_DEVELOPMENT_FALLBACK"


if not raw_entity_metadata:
    raise RuntimeError(
        "The resolved metadata collection is empty."
    )


# =============================================================================
# 4. NORMALISE AND VALIDATE EACH ENTITY
# =============================================================================

required_metadata_fields = {
    "ingestion_id",
    "source_system",
    "source_company",
    "source_object",
    "target_table",
    "load_type",
    "watermark_column",
    "load_sequence",
}

normalised_entities: List[Dict[str, Any]] = []
metadata_errors: List[str] = []


for row_number, raw_entity in enumerate(
    raw_entity_metadata,
    start=1,
):
    if not isinstance(raw_entity, dict):
        metadata_errors.append(
            f"Metadata row {row_number} is not a JSON object."
        )
        continue

    missing_fields = sorted(
        field_name
        for field_name in required_metadata_fields
        if field_name not in raw_entity
    )

    if missing_fields:
        metadata_errors.append(
            f"Metadata row {row_number} is missing fields: "
            + ", ".join(missing_fields)
        )
        continue

    try:
        ingestion_id = normalise_metadata_integer(
            raw_entity.get("ingestion_id"),
            default=-1,
            field_name="ingestion_id",
        )

        source_system = str(
            raw_entity.get("source_system")
        ).strip().upper()

        source_company = str(
            raw_entity.get("source_company")
        ).strip().upper()

        source_object = str(
            raw_entity.get("source_object")
        ).strip()

        source_type = str(
            raw_entity.get(
                "source_type",
                "REST_API",
            )
        ).strip().upper()

        target_schema = validate_identifier(
            raw_entity.get(
                "target_schema",
                "dbo",
            ),
            field_name="target_schema",
            ingestion_id=ingestion_id,
        )

        target_table = validate_identifier(
            raw_entity.get("target_table"),
            field_name="target_table",
            ingestion_id=ingestion_id,
        )

        target_line_table_raw = (
            normalise_nullable_string(
                raw_entity.get(
                    "target_line_table"
                )
            )
        )

        target_line_table = validate_identifier(
            target_line_table_raw,
            field_name="target_line_table",
            ingestion_id=ingestion_id,
            allow_none=True,
        )

        load_type = str(
            raw_entity.get("load_type")
        ).strip().upper()

        watermark_column = str(
            raw_entity.get("watermark_column")
        ).strip()

        last_watermark = parse_optional_datetime(
            raw_entity.get(
                "last_watermark"
            )
        )

        api_query_template = (
            normalise_nullable_string(
                raw_entity.get(
                    "api_query_template"
                )
            )
        )

        response_collection = str(
            raw_entity.get(
                "response_collection",
                source_object,
            )
        ).strip()

        page_size = normalise_metadata_integer(
            raw_entity.get("page_size"),
            default=1000,
            field_name="page_size",
        )

        supports_pagination = (
            normalise_metadata_integer(
                raw_entity.get(
                    "supports_pagination"
                ),
                default=1,
                field_name="supports_pagination",
            )
        )

        load_sequence = (
            normalise_metadata_integer(
                raw_entity.get(
                    "load_sequence"
                ),
                default=0,
                field_name="load_sequence",
            )
        )

        is_active = normalise_metadata_integer(
            raw_entity.get("is_active"),
            default=1,
            field_name="is_active",
        )

        entity = {
            "ingestion_id": ingestion_id,
            "source_system": source_system,
            "source_company": source_company,
            "source_object": source_object,
            "source_type": source_type,
            "target_schema": target_schema,
            "target_table": target_table,
            "target_line_table": target_line_table,
            "load_type": load_type,
            "watermark_column": watermark_column,
            "last_watermark": last_watermark,
            "api_query_template": api_query_template,
            "response_collection": response_collection,
            "page_size": page_size,
            "supports_pagination": supports_pagination,
            "load_sequence": load_sequence,
            "is_active": is_active,
        }

    except (
        TypeError,
        ValueError,
        KeyError,
    ) as exc:
        metadata_errors.append(
            f"Metadata row {row_number} could not be normalised: "
            f"{exc}"
        )
        continue


    # Validate binary flags before filtering.
    if entity["is_active"] not in {0, 1}:
        metadata_errors.append(
            f"Ingestion ID {entity['ingestion_id']} has an invalid "
            "is_active value. Expected 0 or 1."
        )
        continue

    if entity["supports_pagination"] not in {0, 1}:
        metadata_errors.append(
            f"Ingestion ID {entity['ingestion_id']} has an invalid "
            "supports_pagination value. Expected 0 or 1."
        )


    # Skip valid inactive metadata records.
    if entity["is_active"] == 0:
        continue


    # Filter metadata to the current notebook execution context.
    if entity["source_system"] != SOURCE_SYSTEM:
        continue

    if entity["source_company"] != SOURCE_COMPANY:
        continue

    if entity["load_type"] != LOAD_TYPE:
        continue


    # Validate the selected active entity.
    if entity["ingestion_id"] <= 0:
        metadata_errors.append(
            f"Metadata row {row_number} has an invalid ingestion_id."
        )

    if not entity["source_object"]:
        metadata_errors.append(
            f"Ingestion ID {entity['ingestion_id']} has a blank "
            "source_object."
        )

    if (
        entity["source_object"]
        not in SUPPORTED_QBO_SOURCE_OBJECTS
    ):
        metadata_errors.append(
            f"Ingestion ID {entity['ingestion_id']} contains an "
            f"unsupported source object: "
            f"{entity['source_object']!r}."
        )

    if entity["source_type"] not in SUPPORTED_SOURCE_TYPES:
        metadata_errors.append(
            f"Ingestion ID {entity['ingestion_id']} has an "
            f"unsupported source_type: "
            f"{entity['source_type']!r}."
        )

    if entity["load_type"] not in SUPPORTED_ENTITY_LOAD_TYPES:
        metadata_errors.append(
            f"Ingestion ID {entity['ingestion_id']} has an "
            f"unsupported load_type: "
            f"{entity['load_type']!r}."
        )

    if (
        entity["watermark_column"]
        not in SUPPORTED_WATERMARK_COLUMNS
    ):
        metadata_errors.append(
            f"Ingestion ID {entity['ingestion_id']} has an "
            "unsupported watermark column: "
            f"{entity['watermark_column']!r}."
        )

    if not entity["response_collection"]:
        metadata_errors.append(
            f"Ingestion ID {entity['ingestion_id']} has a blank "
            "response_collection."
        )

    if entity["page_size"] <= 0:
        metadata_errors.append(
            f"Ingestion ID {entity['ingestion_id']} has an invalid "
            "page_size. The value must be greater than zero."
        )

    if entity["page_size"] > 1000:
        metadata_errors.append(
            f"Ingestion ID {entity['ingestion_id']} has page_size "
            f"{entity['page_size']}. The configured maximum is 1000."
        )

    if entity["load_sequence"] < 0:
        metadata_errors.append(
            f"Ingestion ID {entity['ingestion_id']} has a negative "
            "load_sequence."
        )

    normalised_entities.append(
        entity
    )


# =============================================================================
# 5. CROSS-ENTITY VALIDATION
# =============================================================================

if metadata_errors:
    raise RuntimeError(
        "Incremental metadata validation failed:\n"
        + "\n".join(
            f"{number}. {message}"
            for number, message in enumerate(
                metadata_errors,
                start=1,
            )
        )
    )


if not normalised_entities:
    raise RuntimeError(
        "No active QBO incremental entities were found for "
        f"source system {SOURCE_SYSTEM!r}, "
        f"company {SOURCE_COMPANY!r} and "
        f"load type {LOAD_TYPE!r}."
    )


ingestion_ids = [
    entity["ingestion_id"]
    for entity in normalised_entities
]

duplicate_ingestion_ids = find_duplicates(
    ingestion_ids
)

if duplicate_ingestion_ids:
    raise RuntimeError(
        "Duplicate ingestion IDs were found: "
        + ", ".join(
            str(value)
            for value in duplicate_ingestion_ids
        )
    )


source_objects_lower = [
    entity["source_object"].lower()
    for entity in normalised_entities
]

duplicate_source_objects = find_duplicates(
    source_objects_lower
)

if duplicate_source_objects:
    raise RuntimeError(
        "Duplicate source objects were found: "
        + ", ".join(
            duplicate_source_objects
        )
    )


target_tables_lower = [
    entity["target_table"].lower()
    for entity in normalised_entities
]

duplicate_target_tables = find_duplicates(
    target_tables_lower
)

if duplicate_target_tables:
    raise RuntimeError(
        "Duplicate target tables were found: "
        + ", ".join(
            duplicate_target_tables
        )
    )


configured_line_tables_lower = [
    entity["target_line_table"].lower()
    for entity in normalised_entities
    if entity["target_line_table"] is not None
]

duplicate_line_tables = find_duplicates(
    configured_line_tables_lower
)

if duplicate_line_tables:
    raise RuntimeError(
        "Duplicate target line tables were found: "
        + ", ".join(
            duplicate_line_tables
        )
    )


all_target_tables = set(
    target_tables_lower
)

header_line_collisions = sorted(
    set(configured_line_tables_lower)
    .intersection(all_target_tables)
)

if header_line_collisions:
    raise RuntimeError(
        "The following tables are configured as both header and line "
        "targets: "
        + ", ".join(header_line_collisions)
    )


load_sequences = [
    entity["load_sequence"]
    for entity in normalised_entities
]

duplicate_load_sequences = find_duplicates(
    load_sequences
)

if duplicate_load_sequences:
    raise RuntimeError(
        "Duplicate load_sequence values were found: "
        + ", ".join(
            str(value)
            for value in duplicate_load_sequences
        )
    )


# =============================================================================
# 6. SORT AND PUBLISH RUNTIME METADATA
# =============================================================================

INCREMENTAL_ENTITIES = sorted(
    normalised_entities,
    key=lambda entity: (
        entity["load_sequence"],
        entity["ingestion_id"],
    ),
)


ENTITY_BY_SOURCE_OBJECT = {
    entity["source_object"]: entity
    for entity in INCREMENTAL_ENTITIES
}


ENTITY_BY_INGESTION_ID = {
    entity["ingestion_id"]: entity
    for entity in INCREMENTAL_ENTITIES
}


ENTITY_BY_TARGET_TABLE = {
    entity["target_table"]: entity
    for entity in INCREMENTAL_ENTITIES
}


# =============================================================================
# 7. BUILD A SAFE METADATA SUMMARY
# =============================================================================

entities_with_watermark_count = sum(
    1
    for entity in INCREMENTAL_ENTITIES
    if entity["last_watermark"] is not None
)


entities_without_watermark_count = sum(
    1
    for entity in INCREMENTAL_ENTITIES
    if entity["last_watermark"] is None
)


entities_with_line_tables_count = sum(
    1
    for entity in INCREMENTAL_ENTITIES
    if entity["target_line_table"] is not None
)


metadata_summary_rows = [
    {
        "ingestion_id": entity["ingestion_id"],
        "source_object": entity["source_object"],
        "target_table": entity["target_table"],
        "target_line_table": entity["target_line_table"],
        "load_type": entity["load_type"],
        "watermark_column": entity["watermark_column"],
        "last_watermark": (
            entity["last_watermark"].isoformat()
            if entity["last_watermark"] is not None
            else None
        ),
        "load_sequence": entity["load_sequence"],
    }
    for entity in INCREMENTAL_ENTITIES
]


metadata_summary_schema = StructType([
    StructField(
        "ingestion_id",
        IntegerType(),
        False,
    ),
    StructField(
        "source_object",
        StringType(),
        False,
    ),
    StructField(
        "target_table",
        StringType(),
        False,
    ),
    StructField(
        "target_line_table",
        StringType(),
        True,
    ),
    StructField(
        "load_type",
        StringType(),
        False,
    ),
    StructField(
        "watermark_column",
        StringType(),
        False,
    ),
    StructField(
        "last_watermark",
        StringType(),
        True,
    ),
    StructField(
        "load_sequence",
        IntegerType(),
        False,
    ),
])


METADATA_SUMMARY_DF = spark.createDataFrame(
    metadata_summary_rows,
    schema=metadata_summary_schema,
)


display(
    METADATA_SUMMARY_DF
    .orderBy(
        "load_sequence"
    )
)


# =============================================================================
# 8. SAFE RUNTIME SUMMARY
# =============================================================================

print("=" * 80)
print("QBO INCREMENTAL METADATA RESOLUTION")
print("=" * 80)
print(f"Metadata source                    : {METADATA_SOURCE}")
print(f"Source system                      : {SOURCE_SYSTEM}")
print(f"Source company                     : {SOURCE_COMPANY}")
print(f"Load type                          : {LOAD_TYPE}")
print(
    "Active incremental entities      : "
    f"{len(INCREMENTAL_ENTITIES)}"
)
print(
    "Entities with existing watermark : "
    f"{entities_with_watermark_count}"
)
print(
    "Entities requiring initial seed   : "
    f"{entities_without_watermark_count}"
)
print(
    "Entities with line tables         : "
    f"{entities_with_line_tables_count}"
)
print(
    "First entity                      : "
    f"{INCREMENTAL_ENTITIES[0]['source_object']}"
)
print(
    "Last entity                       : "
    f"{INCREMENTAL_ENTITIES[-1]['source_object']}"
)
print("Incremental metadata validation   : SUCCEEDED")
print("=" * 80)






for entity in INCREMENTAL_ENTITIES:
    source_object = str(
        entity.get("source_object", "")
    ).strip()

    ingestion_id = entity.get("ingestion_id")

    try:
        previous_watermark_utc = (
            normalise_watermark_utc(
                entity.get("last_watermark")
            )
        )

    except (TypeError, ValueError) as exc:
        watermark_errors.append(
            f"{source_object}: invalid previous watermark. {exc}"
        )
        continue

    if previous_watermark_utc is None:
        base_watermark_utc = (
            EXTRACTION_END_UTC
            - timedelta(days=INITIAL_LOOKBACK_DAYS)
        )

        watermark_mode = "INITIAL_SEED"

    else:
        base_watermark_utc = previous_watermark_utc
        watermark_mode = "EXISTING_WATERMARK"

    extraction_start_utc = (
        base_watermark_utc
        - timedelta(
            minutes=WATERMARK_OVERLAP_MINUTES
        )
    )

StatementMeta(, 6bf6e71d-4576-4274-981e-6ccc62d6e773, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5af2b2e9-3970-402b-b22e-89f28520df51)

QBO INCREMENTAL METADATA RESOLUTION
Metadata source                    : MANUAL_DEVELOPMENT_FALLBACK
Source system                      : QBO
Source company                     : ES01
Load type                          : INCREMENTAL
Active incremental entities      : 15
Entities with existing watermark : 0
Entities requiring initial seed   : 15
Entities with line tables         : 7
First entity                      : Account
Last entity                       : JournalEntry
Incremental metadata validation   : SUCCEEDED


In [1]:
# =============================================================================
# CELL 5A — RESOLVE INCREMENTAL WATERMARK WINDOWS
# QuickBooks Online Incremental Bronze Ingestion
#
# Purpose:
# 1. Resolve one extraction window for every active entity.
# 2. Apply a configurable overlap to protect timestamp boundaries.
# 3. Seed new entities within the QuickBooks CDC lookback limit.
# 4. Publish a validated runtime DataFrame for CDC request generation.
#
# Prerequisite:
# Cell 4 must create INCREMENTAL_ENTITIES.
#
# Metadata requirement:
# last_watermark must be supplied by the pipeline Lookup through a LEFT JOIN
# between ctl.ingestion_config and ctl.watermark_tracker.
# =============================================================================


# -----------------------------------------------------------------------------
# Imports
# -----------------------------------------------------------------------------

from datetime import datetime, timedelta, timezone
from typing import Any, Dict, List, Optional

from pyspark.sql import functions as F
from pyspark.sql.types import (
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)


# -----------------------------------------------------------------------------
# Constants
# -----------------------------------------------------------------------------

# QuickBooks CDC supports a maximum lookback of 30 days.
# Use 29 days to stay safely inside that boundary.
INITIAL_LOOKBACK_DAYS = 29

MINIMUM_OVERLAP_MINUTES = 0
MAXIMUM_OVERLAP_MINUTES = 60


# =============================================================================
# 1. VALIDATE UPSTREAM RUNTIME OBJECTS
# =============================================================================

required_runtime_objects = [
    "PIPELINE_RUN_ID",
    "WATERMARK_OVERLAP_MINUTES",
    "INCREMENTAL_ENTITIES",
]

missing_runtime_objects = [
    object_name
    for object_name in required_runtime_objects
    if object_name not in globals()
]

if missing_runtime_objects:
    raise NameError(
        "Cell 5A cannot run because these upstream objects are missing: "
        + ", ".join(missing_runtime_objects)
        + ". Run Cells 1 through 4 first."
    )


if not isinstance(INCREMENTAL_ENTITIES, list):
    raise TypeError(
        "INCREMENTAL_ENTITIES must be a Python list."
    )


if not INCREMENTAL_ENTITIES:
    raise RuntimeError(
        "INCREMENTAL_ENTITIES is empty."
    )


try:
    WATERMARK_OVERLAP_MINUTES = int(
        WATERMARK_OVERLAP_MINUTES
    )

except (TypeError, ValueError) as exc:
    raise ValueError(
        "WATERMARK_OVERLAP_MINUTES must be a valid integer."
    ) from exc


if not (
    MINIMUM_OVERLAP_MINUTES
    <= WATERMARK_OVERLAP_MINUTES
    <= MAXIMUM_OVERLAP_MINUTES
):
    raise ValueError(
        "WATERMARK_OVERLAP_MINUTES must be between "
        f"{MINIMUM_OVERLAP_MINUTES} and "
        f"{MAXIMUM_OVERLAP_MINUTES}."
    )


# =============================================================================
# 2. FIX ONE EXTRACTION END TIME FOR THE COMPLETE RUN
# =============================================================================

# Every entity receives the same upper boundary. This provides a consistent,
# repeatable extraction window across the complete pipeline run.
EXTRACTION_END_UTC = datetime.now(
    timezone.utc
)


# =============================================================================
# 3. WATERMARK HELPER FUNCTIONS
# =============================================================================

def normalise_watermark_utc(
    value: Any,
) -> Optional[datetime]:
    """
    Convert a supported watermark value to a timezone-aware UTC datetime.

    Supported input:
    - None
    - Python datetime
    - ISO-formatted timestamp string
    - SQL timestamp string
    """
    if value is None:
        return None

    if isinstance(value, datetime):
        parsed_value = value

    else:
        value_text = str(value).strip()

        if (
            not value_text
            or value_text.upper() == "NULL"
        ):
            return None

        normalised_text = value_text.replace(
            "Z",
            "+00:00",
        )

        try:
            parsed_value = datetime.fromisoformat(
                normalised_text
            )

        except ValueError:
            supported_formats = [
                "%Y-%m-%d %H:%M:%S.%f",
                "%Y-%m-%d %H:%M:%S",
                "%Y-%m-%d",
            ]

            parsed_value = None

            for datetime_format in supported_formats:
                try:
                    parsed_value = datetime.strptime(
                        value_text,
                        datetime_format,
                    )
                    break

                except ValueError:
                    continue

            if parsed_value is None:
                raise ValueError(
                    "Unable to parse watermark value: "
                    f"{value!r}"
                )

    if parsed_value.tzinfo is None:
        parsed_value = parsed_value.replace(
            tzinfo=timezone.utc
        )

    return parsed_value.astimezone(
        timezone.utc
    )


def spark_utc_timestamp(
    value: Optional[datetime],
) -> Optional[datetime]:
    """
    Convert a timezone-aware UTC datetime into a timezone-naive UTC value
    suitable for Spark TimestampType construction.
    """
    if value is None:
        return None

    return (
        value
        .astimezone(timezone.utc)
        .replace(tzinfo=None)
    )


# =============================================================================
# 4. BUILD RUNTIME WATERMARK WINDOWS
# =============================================================================

watermark_runtime_rows: List[
    Dict[str, Any]
] = []

watermark_errors: List[str] = []


for entity in INCREMENTAL_ENTITIES:
    source_object = str(
        entity.get("source_object", "")
    ).strip()

    ingestion_id = entity.get(
        "ingestion_id"
    )

    if not source_object:
        watermark_errors.append(
            f"Ingestion ID {ingestion_id!r} has a blank source_object."
        )
        continue

    try:
        previous_watermark_utc = (
            normalise_watermark_utc(
                entity.get("last_watermark")
            )
        )

    except (TypeError, ValueError) as exc:
        watermark_errors.append(
            f"{source_object}: invalid previous watermark. {exc}"
        )
        continue


    if previous_watermark_utc is None:
        base_watermark_utc = (
            EXTRACTION_END_UTC
            - timedelta(
                days=INITIAL_LOOKBACK_DAYS
            )
        )

        watermark_mode = "INITIAL_SEED"

    else:
        base_watermark_utc = (
            previous_watermark_utc
        )

        watermark_mode = "EXISTING_WATERMARK"


    extraction_start_utc = (
        base_watermark_utc
        - timedelta(
            minutes=WATERMARK_OVERLAP_MINUTES
        )
    )


    # QuickBooks CDC cannot process a start point more than 30 days old.
    minimum_supported_start_utc = (
        EXTRACTION_END_UTC
        - timedelta(days=30)
    )

    if extraction_start_utc < minimum_supported_start_utc:
        extraction_start_utc = (
            minimum_supported_start_utc
        )

        watermark_mode = (
            "CLAMPED_TO_CDC_LIMIT"
            if previous_watermark_utc is not None
            else "INITIAL_SEED"
        )


    if extraction_start_utc >= EXTRACTION_END_UTC:
        watermark_errors.append(
            f"{source_object}: extraction_start_utc must be earlier "
            "than extraction_end_utc."
        )
        continue


    if (
        previous_watermark_utc is not None
        and previous_watermark_utc > EXTRACTION_END_UTC
    ):
        watermark_errors.append(
            f"{source_object}: the stored watermark is later than the "
            "current extraction end time."
        )
        continue


    watermark_runtime_rows.append(
        {
            "pipeline_run_id": str(
                PIPELINE_RUN_ID
            ),
            "ingestion_id": int(
                entity["ingestion_id"]
            ),
            "source_system": str(
                entity["source_system"]
            ),
            "source_company": str(
                entity["source_company"]
            ),
            "source_object": source_object,
            "target_table": str(
                entity["target_table"]
            ),
            "target_line_table": (
                entity.get(
                    "target_line_table"
                )
            ),
            "watermark_column": str(
                entity["watermark_column"]
            ),
            "previous_watermark_utc": (
                spark_utc_timestamp(
                    previous_watermark_utc
                )
            ),
            "base_watermark_utc": (
                spark_utc_timestamp(
                    base_watermark_utc
                )
            ),
            "extraction_start_utc": (
                spark_utc_timestamp(
                    extraction_start_utc
                )
            ),
            "extraction_end_utc": (
                spark_utc_timestamp(
                    EXTRACTION_END_UTC
                )
            ),
            "watermark_mode": (
                watermark_mode
            ),
            "watermark_overlap_minutes": (
                WATERMARK_OVERLAP_MINUTES
            ),
            "load_sequence": int(
                entity["load_sequence"]
            ),
        }
    )


if watermark_errors:
    raise RuntimeError(
        "Incremental watermark resolution failed:\n"
        + "\n".join(
            f"{number}. {message}"
            for number, message in enumerate(
                watermark_errors,
                start=1,
            )
        )
    )


# =============================================================================
# 5. DEFINE THE EXPLICIT SPARK SCHEMA
# =============================================================================

watermark_runtime_schema = StructType([
    StructField(
        "pipeline_run_id",
        StringType(),
        False,
    ),
    StructField(
        "ingestion_id",
        IntegerType(),
        False,
    ),
    StructField(
        "source_system",
        StringType(),
        False,
    ),
    StructField(
        "source_company",
        StringType(),
        False,
    ),
    StructField(
        "source_object",
        StringType(),
        False,
    ),
    StructField(
        "target_table",
        StringType(),
        False,
    ),
    StructField(
        "target_line_table",
        StringType(),
        True,
    ),
    StructField(
        "watermark_column",
        StringType(),
        False,
    ),
    StructField(
        "previous_watermark_utc",
        TimestampType(),
        True,
    ),
    StructField(
        "base_watermark_utc",
        TimestampType(),
        False,
    ),
    StructField(
        "extraction_start_utc",
        TimestampType(),
        False,
    ),
    StructField(
        "extraction_end_utc",
        TimestampType(),
        False,
    ),
    StructField(
        "watermark_mode",
        StringType(),
        False,
    ),
    StructField(
        "watermark_overlap_minutes",
        IntegerType(),
        False,
    ),
    StructField(
        "load_sequence",
        IntegerType(),
        False,
    ),
])


# =============================================================================
# 6. CREATE AND VALIDATE THE RUNTIME DATAFRAME
# =============================================================================

WATERMARK_RUNTIME_DF = spark.createDataFrame(
    watermark_runtime_rows,
    schema=watermark_runtime_schema,
)


runtime_entity_count = (
    WATERMARK_RUNTIME_DF.count()
)


if runtime_entity_count != len(
    INCREMENTAL_ENTITIES
):
    raise RuntimeError(
        "Watermark runtime row count does not match the incremental "
        "metadata entity count. "
        f"Expected {len(INCREMENTAL_ENTITIES)}, "
        f"created {runtime_entity_count}."
    )


duplicate_runtime_entity_count = (
    WATERMARK_RUNTIME_DF
    .groupBy(
        "source_system",
        "source_company",
        "source_object",
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)


if duplicate_runtime_entity_count > 0:
    raise RuntimeError(
        "Duplicate entity watermark windows were created."
    )


invalid_window_count = (
    WATERMARK_RUNTIME_DF
    .filter(
        F.col("extraction_start_utc")
        >= F.col("extraction_end_utc")
    )
    .count()
)


if invalid_window_count > 0:
    raise RuntimeError(
        "One or more invalid extraction windows were created."
    )


# =============================================================================
# 7. CALCULATE SAFE SUMMARY METRICS
# =============================================================================

initial_seed_count = (
    WATERMARK_RUNTIME_DF
    .filter(
        F.col("watermark_mode")
        == "INITIAL_SEED"
    )
    .count()
)


existing_watermark_count = (
    WATERMARK_RUNTIME_DF
    .filter(
        F.col("watermark_mode")
        == "EXISTING_WATERMARK"
    )
    .count()
)


clamped_watermark_count = (
    WATERMARK_RUNTIME_DF
    .filter(
        F.col("watermark_mode")
        == "CLAMPED_TO_CDC_LIMIT"
    )
    .count()
)


minimum_extraction_start = (
    WATERMARK_RUNTIME_DF
    .agg(
        F.min(
            "extraction_start_utc"
        ).alias(
            "minimum_start"
        )
    )
    .first()["minimum_start"]
)


maximum_extraction_start = (
    WATERMARK_RUNTIME_DF
    .agg(
        F.max(
            "extraction_start_utc"
        ).alias(
            "maximum_start"
        )
    )
    .first()["maximum_start"]
)


# =============================================================================
# 8. DISPLAY THE RUNTIME WATERMARK MANIFEST
# =============================================================================

display(
    WATERMARK_RUNTIME_DF
    .orderBy(
        "load_sequence"
    )
)


# =============================================================================
# 9. SAFE RUNTIME SUMMARY
# =============================================================================

print("=" * 80)
print("INCREMENTAL WATERMARK RESOLUTION")
print("=" * 80)
print(f"Pipeline run ID              : {PIPELINE_RUN_ID}")
print(f"Runtime entities             : {runtime_entity_count}")
print(f"Initial watermark seeds      : {initial_seed_count}")
print(f"Existing watermarks          : {existing_watermark_count}")
print(f"Watermarks clamped to limit  : {clamped_watermark_count}")
print(
    "Extraction end UTC          : "
    f"{EXTRACTION_END_UTC.isoformat()}"
)
print(
    "Minimum extraction start    : "
    f"{minimum_extraction_start}"
)
print(
    "Maximum extraction start    : "
    f"{maximum_extraction_start}"
)
print(
    "Watermark overlap minutes   : "
    f"{WATERMARK_OVERLAP_MINUTES}"
)
print("Watermark resolution         : SUCCEEDED")
print("=" * 80)


StatementMeta(, 512b264d-925e-4515-a439-f76b522e9d96, 3, Finished, Available, Finished, False)

NameError: Cell 5A cannot run because these upstream objects are missing: PIPELINE_RUN_ID, WATERMARK_OVERLAP_MINUTES, INCREMENTAL_ENTITIES. Run Cells 1 through 4 first.

In [ ]:
# =============================================================================
# CELL 5B — BUILD QUICKBOOKS CDC REQUEST MANIFEST
# QuickBooks Online Incremental Bronze Ingestion
#
# Purpose:
# 1. Convert the watermark windows produced by Cell 5A into CDC requests.
# 2. Build one independently auditable request per QuickBooks entity.
# 3. Validate the complete request manifest before any API request is sent.
# 4. Publish Spark and Python representations for the extraction cell.
#
# Prerequisite:
# Cell 5A must create WATERMARK_RUNTIME_DF.
#
# Design decision:
# One entity is used per CDC request to support entity-level watermarks,
# auditing, retry handling and failure isolation.
# =============================================================================


# -----------------------------------------------------------------------------
# Imports
# -----------------------------------------------------------------------------

from datetime import datetime, timezone
from typing import Any, Dict, List
from urllib.parse import urlencode, urlparse

from pyspark.sql import functions as F
from pyspark.sql.types import (
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)


# -----------------------------------------------------------------------------
# Constants
# -----------------------------------------------------------------------------

CDC_ENTITIES_PER_REQUEST = 1

EXPECTED_RESPONSE_CONTENT_TYPE = (
    "application/json"
)

READY_REQUEST_STATUS = "READY"


# =============================================================================
# 1. VALIDATE REQUIRED UPSTREAM OBJECTS
# =============================================================================

required_runtime_objects = [
    "PIPELINE_RUN_ID",
    "QBO_API_BASE_URL",
    "QBO_REALM_ID",
    "QBO_MINOR_VERSION",
    "WATERMARK_RUNTIME_DF",
    "INCREMENTAL_ENTITIES",
]

missing_runtime_objects = [
    object_name
    for object_name in required_runtime_objects
    if object_name not in globals()
]

if missing_runtime_objects:
    raise NameError(
        "Cell 5B cannot run because these upstream objects are missing: "
        + ", ".join(missing_runtime_objects)
        + ". Run Cells 1 through 5A first."
    )


if not isinstance(
    INCREMENTAL_ENTITIES,
    list,
):
    raise TypeError(
        "INCREMENTAL_ENTITIES must be a Python list."
    )


if not INCREMENTAL_ENTITIES:
    raise RuntimeError(
        "INCREMENTAL_ENTITIES is empty."
    )


if WATERMARK_RUNTIME_DF is None:
    raise RuntimeError(
        "WATERMARK_RUNTIME_DF is not available."
    )


# =============================================================================
# 2. VALIDATE QUICKBOOKS CDC CONFIGURATION
# =============================================================================

qbo_api_base_url = str(
    QBO_API_BASE_URL
).strip().rstrip("/")

qbo_realm_id = str(
    QBO_REALM_ID
).strip()

qbo_minor_version = str(
    QBO_MINOR_VERSION
).strip()


if not qbo_api_base_url:
    raise ValueError(
        "QBO_API_BASE_URL cannot be blank."
    )


if not qbo_realm_id:
    raise ValueError(
        "QBO_REALM_ID cannot be blank."
    )


if not qbo_realm_id.isdigit():
    raise ValueError(
        "QBO_REALM_ID must contain only numeric characters."
    )


if not qbo_minor_version:
    raise ValueError(
        "QBO_MINOR_VERSION cannot be blank."
    )


if not qbo_minor_version.isdigit():
    raise ValueError(
        "QBO_MINOR_VERSION must contain only numeric characters."
    )


parsed_api_url = urlparse(
    qbo_api_base_url
)

if parsed_api_url.scheme != "https":
    raise ValueError(
        "QBO_API_BASE_URL must use HTTPS."
    )


if not parsed_api_url.hostname:
    raise ValueError(
        "QBO_API_BASE_URL does not contain a valid hostname."
    )


CDC_ENDPOINT = (
    f"{qbo_api_base_url}/v3/company/"
    f"{qbo_realm_id}/cdc"
)


# =============================================================================
# 3. HELPER FUNCTIONS
# =============================================================================

def require_utc_datetime(
    value: Any,
    field_name: str,
    source_object: str,
) -> datetime:
    """
    Validate a required datetime and return it as a timezone-aware
    UTC datetime.

    Spark TimestampType values returned by collect() may not retain
    timezone information, so timezone-naive values are interpreted as UTC.
    """
    if value is None:
        raise ValueError(
            f"{field_name} is null for {source_object}."
        )

    if not isinstance(
        value,
        datetime,
    ):
        raise TypeError(
            f"{field_name} for {source_object} must be a datetime. "
            f"Received: {type(value).__name__}."
        )

    if value.tzinfo is None:
        return value.replace(
            tzinfo=timezone.utc
        )

    return value.astimezone(
        timezone.utc
    )


def spark_utc_timestamp(
    value: datetime,
) -> datetime:
    """
    Convert a timezone-aware UTC datetime into a timezone-naive UTC
    datetime suitable for Spark TimestampType.
    """
    return (
        value
        .astimezone(timezone.utc)
        .replace(tzinfo=None)
    )


def format_qbo_datetime(
    value: datetime,
    source_object: str,
) -> str:
    """
    Format a UTC datetime for the QuickBooks changedSince parameter.

    Example:
    2026-08-04T16:06:01Z
    """
    value_utc = require_utc_datetime(
        value=value,
        field_name="changedSince datetime",
        source_object=source_object,
    )

    return value_utc.strftime(
        "%Y-%m-%dT%H:%M:%SZ"
    )


def build_cdc_query_parameters(
    entity_name: str,
    changed_since: str,
    minor_version: str,
) -> Dict[str, str]:
    """Build the query parameters used by the CDC API request."""
    return {
        "entities": entity_name,
        "changedSince": changed_since,
        "minorversion": minor_version,
    }


def build_cdc_request_url(
    endpoint: str,
    query_parameters: Dict[str, str],
) -> str:
    """
    Build an encoded request URL for safe diagnostics and auditing.

    The execution cell should still call requests.get() using params=
    rather than manually assembling the request URL.
    """
    return (
        endpoint
        + "?"
        + urlencode(query_parameters)
    )


# =============================================================================
# 4. BUILD THE ENTITY METADATA LOOKUP
# =============================================================================

entity_metadata_lookup = {
    str(entity["source_object"]).strip(): entity
    for entity in INCREMENTAL_ENTITIES
}


if len(entity_metadata_lookup) != len(
    INCREMENTAL_ENTITIES
):
    raise RuntimeError(
        "Duplicate source_object values exist in "
        "INCREMENTAL_ENTITIES."
    )


# =============================================================================
# 5. COLLECT AND VALIDATE WATERMARK RUNTIME ROWS
# =============================================================================

runtime_rows = (
    WATERMARK_RUNTIME_DF
    .orderBy(
        "load_sequence",
        "ingestion_id",
    )
    .collect()
)


runtime_entity_count = len(
    runtime_rows
)


if runtime_entity_count == 0:
    raise RuntimeError(
        "WATERMARK_RUNTIME_DF contains no entities."
    )


if runtime_entity_count != len(
    INCREMENTAL_ENTITIES
):
    raise RuntimeError(
        "The watermark runtime count does not match the metadata count. "
        f"Runtime rows: {runtime_entity_count}; "
        f"metadata entities: {len(INCREMENTAL_ENTITIES)}."
    )


# =============================================================================
# 6. BUILD ONE CDC REQUEST PER ENTITY
# =============================================================================

cdc_request_rows: List[
    Dict[str, Any]
] = []

request_errors: List[str] = []


for request_sequence, runtime_row in enumerate(
    runtime_rows,
    start=1,
):
    source_object = str(
        runtime_row["source_object"]
        or ""
    ).strip()

    try:
        if not source_object:
            raise ValueError(
                "source_object cannot be blank."
            )


        entity_metadata = (
            entity_metadata_lookup.get(
                source_object
            )
        )

        if entity_metadata is None:
            raise KeyError(
                "No matching entity metadata was found."
            )


        extraction_start_utc = (
            require_utc_datetime(
                value=runtime_row[
                    "extraction_start_utc"
                ],
                field_name=(
                    "extraction_start_utc"
                ),
                source_object=source_object,
            )
        )


        extraction_end_utc = (
            require_utc_datetime(
                value=runtime_row[
                    "extraction_end_utc"
                ],
                field_name=(
                    "extraction_end_utc"
                ),
                source_object=source_object,
            )
        )


        if (
            extraction_start_utc
            >= extraction_end_utc
        ):
            raise ValueError(
                "Extraction start must be earlier than extraction end."
            )


        changed_since = format_qbo_datetime(
            value=extraction_start_utc,
            source_object=source_object,
        )


        response_collection = str(
            entity_metadata.get(
                "response_collection",
                source_object,
            )
            or ""
        ).strip()


        if not response_collection:
            raise ValueError(
                "response_collection cannot be blank."
            )


        query_parameters = (
            build_cdc_query_parameters(
                entity_name=source_object,
                changed_since=changed_since,
                minor_version=qbo_minor_version,
            )
        )


        request_url = build_cdc_request_url(
            endpoint=CDC_ENDPOINT,
            query_parameters=query_parameters,
        )


        target_line_table = runtime_row[
            "target_line_table"
        ]

        if target_line_table is not None:
            target_line_table = str(
                target_line_table
            ).strip() or None


        cdc_request_rows.append(
            {
                "pipeline_run_id": str(
                    PIPELINE_RUN_ID
                ),
                "request_sequence": int(
                    request_sequence
                ),
                "ingestion_id": int(
                    runtime_row[
                        "ingestion_id"
                    ]
                ),
                "source_system": str(
                    runtime_row[
                        "source_system"
                    ]
                ),
                "source_company": str(
                    runtime_row[
                        "source_company"
                    ]
                ),
                "source_object": (
                    source_object
                ),
                "response_collection": (
                    response_collection
                ),
                "target_table": str(
                    runtime_row[
                        "target_table"
                    ]
                ),
                "target_line_table": (
                    target_line_table
                ),
                "watermark_mode": str(
                    runtime_row[
                        "watermark_mode"
                    ]
                ),
                "previous_watermark_utc": (
                    runtime_row[
                        "previous_watermark_utc"
                    ]
                ),
                "extraction_start_utc": (
                    spark_utc_timestamp(
                        extraction_start_utc
                    )
                ),
                "extraction_end_utc": (
                    spark_utc_timestamp(
                        extraction_end_utc
                    )
                ),
                "changed_since_parameter": (
                    changed_since
                ),
                "cdc_endpoint": (
                    CDC_ENDPOINT
                ),
                "request_url": (
                    request_url
                ),
                "minor_version": (
                    qbo_minor_version
                ),
                "expected_content_type": (
                    EXPECTED_RESPONSE_CONTENT_TYPE
                ),
                "request_status": (
                    READY_REQUEST_STATUS
                ),
            }
        )


    except Exception as exc:
        request_errors.append(
            f"{source_object or '[blank entity]'}: {exc}"
        )


# =============================================================================
# 7. FAIL BEFORE EXTRACTION WHEN THE MANIFEST IS INVALID
# =============================================================================

if request_errors:
    raise RuntimeError(
        "CDC request-manifest construction failed:\n"
        + "\n".join(
            f"{error_number}. {error_text}"
            for error_number, error_text in enumerate(
                request_errors,
                start=1,
            )
        )
    )


if not cdc_request_rows:
    raise RuntimeError(
        "No QuickBooks CDC requests were created."
    )


# =============================================================================
# 8. DEFINE THE EXPLICIT MANIFEST SCHEMA
# =============================================================================

cdc_request_schema = StructType([
    StructField(
        "pipeline_run_id",
        StringType(),
        False,
    ),
    StructField(
        "request_sequence",
        IntegerType(),
        False,
    ),
    StructField(
        "ingestion_id",
        IntegerType(),
        False,
    ),
    StructField(
        "source_system",
        StringType(),
        False,
    ),
    StructField(
        "source_company",
        StringType(),
        False,
    ),
    StructField(
        "source_object",
        StringType(),
        False,
    ),
    StructField(
        "response_collection",
        StringType(),
        False,
    ),
    StructField(
        "target_table",
        StringType(),
        False,
    ),
    StructField(
        "target_line_table",
        StringType(),
        True,
    ),
    StructField(
        "watermark_mode",
        StringType(),
        False,
    ),
    StructField(
        "previous_watermark_utc",
        TimestampType(),
        True,
    ),
    StructField(
        "extraction_start_utc",
        TimestampType(),
        False,
    ),
    StructField(
        "extraction_end_utc",
        TimestampType(),
        False,
    ),
    StructField(
        "changed_since_parameter",
        StringType(),
        False,
    ),
    StructField(
        "cdc_endpoint",
        StringType(),
        False,
    ),
    StructField(
        "request_url",
        StringType(),
        False,
    ),
    StructField(
        "minor_version",
        StringType(),
        False,
    ),
    StructField(
        "expected_content_type",
        StringType(),
        False,
    ),
    StructField(
        "request_status",
        StringType(),
        False,
    ),
])


# =============================================================================
# 9. CREATE AND PUBLISH THE CDC REQUEST MANIFEST
# =============================================================================

CDC_REQUEST_MANIFEST_DF = spark.createDataFrame(
    cdc_request_rows,
    schema=cdc_request_schema,
)


CDC_REQUEST_MANIFEST = [
    row.asDict(
        recursive=True
    )
    for row in (
        CDC_REQUEST_MANIFEST_DF
        .orderBy(
            "request_sequence"
        )
        .collect()
    )
]


# =============================================================================
# 10. VALIDATE THE PUBLISHED MANIFEST
# =============================================================================

manifest_count = len(
    CDC_REQUEST_MANIFEST
)

expected_manifest_count = len(
    INCREMENTAL_ENTITIES
)


if manifest_count != expected_manifest_count:
    raise RuntimeError(
        "CDC request count does not match the number of incremental "
        "entities. "
        f"Expected {expected_manifest_count}, "
        f"created {manifest_count}."
    )


duplicate_request_sequence_count = (
    CDC_REQUEST_MANIFEST_DF
    .groupBy(
        "request_sequence"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)


if duplicate_request_sequence_count > 0:
    raise RuntimeError(
        "The CDC request manifest contains duplicate request sequences."
    )


duplicate_entity_count = (
    CDC_REQUEST_MANIFEST_DF
    .groupBy(
        "source_system",
        "source_company",
        "source_object",
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)


if duplicate_entity_count > 0:
    raise RuntimeError(
        "The CDC request manifest contains duplicate entity requests."
    )


not_ready_count = (
    CDC_REQUEST_MANIFEST_DF
    .filter(
        F.col("request_status")
        != READY_REQUEST_STATUS
    )
    .count()
)


if not_ready_count > 0:
    raise RuntimeError(
        "One or more CDC requests are not READY."
    )


invalid_manifest_window_count = (
    CDC_REQUEST_MANIFEST_DF
    .filter(
        F.col("extraction_start_utc")
        >= F.col("extraction_end_utc")
    )
    .count()
)


if invalid_manifest_window_count > 0:
    raise RuntimeError(
        "One or more CDC manifest rows contain an invalid "
        "extraction window."
    )


null_required_field_count = (
    CDC_REQUEST_MANIFEST_DF
    .filter(
        F.col("pipeline_run_id").isNull()
        | F.col("source_object").isNull()
        | F.col("target_table").isNull()
        | F.col("changed_since_parameter").isNull()
        | F.col("request_url").isNull()
    )
    .count()
)


if null_required_field_count > 0:
    raise RuntimeError(
        "The CDC request manifest contains null mandatory values."
    )


# =============================================================================
# 11. DISPLAY THE SAFE REQUEST MANIFEST
# =============================================================================

display(
    CDC_REQUEST_MANIFEST_DF
    .select(
        "request_sequence",
        "ingestion_id",
        "source_object",
        "target_table",
        "watermark_mode",
        "changed_since_parameter",
        "extraction_end_utc",
        "request_status",
    )
    .orderBy(
        "request_sequence"
    )
)


# =============================================================================
# 12. SAFE EXECUTION SUMMARY
# =============================================================================

first_changed_since = (
    CDC_REQUEST_MANIFEST[0][
        "changed_since_parameter"
    ]
)

last_changed_since = (
    CDC_REQUEST_MANIFEST[-1][
        "changed_since_parameter"
    ]
)


print("=" * 80)
print("QUICKBOOKS CDC REQUEST MANIFEST")
print("=" * 80)
print(f"Pipeline run ID       : {PIPELINE_RUN_ID}")
print(f"CDC endpoint          : {CDC_ENDPOINT}")
print(f"Requests created      : {manifest_count}")
print(
    "Entities per request : "
    f"{CDC_ENTITIES_PER_REQUEST}"
)
print(
    "Distinct entities    : "
    f"{manifest_count - duplicate_entity_count}"
)
print(
    "Requests not ready   : "
    f"{not_ready_count}"
)
print(
    "Expected content type: "
    f"{EXPECTED_RESPONSE_CONTENT_TYPE}"
)
print(
    "First changedSince   : "
    f"{first_changed_since}"
)
print(
    "Last changedSince    : "
    f"{last_changed_since}"
)
print("CDC request manifest : SUCCEEDED")
print("=" * 80)

StatementMeta(, 512b264d-925e-4515-a439-f76b522e9d96, -1, Cancelled, , Cancelled, True)

In [ ]:
# =============================================================================
# CELL 6 — EXECUTE QUICKBOOKS CDC REQUESTS
# QuickBooks Online Incremental Bronze Ingestion
#
# Purpose:
# 1. Execute every READY request in the CDC request manifest.
# 2. Preserve the complete raw QuickBooks response in the Bronze Files area.
# 3. Separate changed and deleted source records.
# 4. Capture entity-level request metrics and errors.
# 5. Publish validated runtime objects for the Bronze MERGE cell.
#
# Prerequisite:
# Cell 5B must create CDC_REQUEST_MANIFEST.
#
# Security:
# Never print the access token or the Authorization request header.
# =============================================================================


# -----------------------------------------------------------------------------
# Imports
# -----------------------------------------------------------------------------

import json
import re
import time
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

import requests
from pyspark.sql import functions as F
from pyspark.sql.types import (
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)


# -----------------------------------------------------------------------------
# Constants
# -----------------------------------------------------------------------------

CDC_MAXIMUM_ATTEMPTS = 3
CDC_REQUEST_TIMEOUT_SECONDS = 120
CDC_INITIAL_RETRY_DELAY_SECONDS = 2
CDC_MAXIMUM_RETRY_DELAY_SECONDS = 60

CDC_TRANSIENT_HTTP_STATUS_CODES = {
    408,
    425,
    429,
    500,
    502,
    503,
    504,
}

CDC_SUCCESS_HTTP_STATUS_CODE = 200
CDC_READY_STATUS = "READY"

SAFE_PATH_COMPONENT_PATTERN = re.compile(
    r"[^a-z0-9_-]+"
)


# =============================================================================
# 1. VALIDATE REQUIRED UPSTREAM OBJECTS
# =============================================================================

required_objects = [
    "CDC_REQUEST_MANIFEST",
    "QBO_REQUEST_HEADERS",
    "PIPELINE_RUN_ID",
    "FAIL_ON_ENTITY_ERROR",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Cell 6 cannot run because these upstream objects are missing: "
        + ", ".join(missing_objects)
        + ". Run Cells 1 through 5B first."
    )


if not isinstance(
    CDC_REQUEST_MANIFEST,
    list,
):
    raise TypeError(
        "CDC_REQUEST_MANIFEST must be a Python list."
    )


if not CDC_REQUEST_MANIFEST:
    raise RuntimeError(
        "CDC_REQUEST_MANIFEST is empty."
    )


if not isinstance(
    QBO_REQUEST_HEADERS,
    dict,
):
    raise TypeError(
        "QBO_REQUEST_HEADERS must be a Python dictionary."
    )


required_header_names = {
    "Authorization",
    "Accept",
}

missing_header_names = sorted(
    header_name
    for header_name in required_header_names
    if not str(
        QBO_REQUEST_HEADERS.get(
            header_name,
            ""
        )
    ).strip()
)

if missing_header_names:
    raise RuntimeError(
        "QBO_REQUEST_HEADERS is missing required values: "
        + ", ".join(missing_header_names)
    )


if not str(PIPELINE_RUN_ID).strip():
    raise ValueError(
        "PIPELINE_RUN_ID cannot be blank."
    )


# Check the token safety boundary created by Cell 3.
if (
    "ACCESS_TOKEN_REFRESH_BY_UTC" in globals()
    and ACCESS_TOKEN_REFRESH_BY_UTC is not None
):
    current_utc_for_token_check = datetime.now(
        timezone.utc
    )

    if current_utc_for_token_check >= ACCESS_TOKEN_REFRESH_BY_UTC:
        raise RuntimeError(
            "The QuickBooks access token has reached its configured "
            "refresh-by time. Rerun Cell 3 before executing CDC requests."
        )


# =============================================================================
# 2. HELPER FUNCTIONS
# =============================================================================

def cdc_utc_now() -> datetime:
    """Return the current timezone-aware UTC timestamp."""
    return datetime.now(
        timezone.utc
    )


def spark_compatible_utc(
    value: Optional[datetime],
) -> Optional[datetime]:
    """
    Convert a datetime into a timezone-naive UTC value suitable for
    Spark TimestampType DataFrame construction.
    """
    if value is None:
        return None

    if not isinstance(
        value,
        datetime,
    ):
        raise TypeError(
            "A Spark timestamp value must be a datetime or None."
        )

    if value.tzinfo is None:
        return value

    return (
        value
        .astimezone(timezone.utc)
        .replace(tzinfo=None)
    )


def safe_response_payload(
    response: requests.Response,
) -> Dict[str, Any]:
    """
    Parse an HTTP response as JSON.

    Response text is truncated if QuickBooks returns non-JSON content.
    No request headers or credentials are included in the error.
    """
    try:
        payload = response.json()

    except ValueError as exc:
        raise RuntimeError(
            "QuickBooks returned a non-JSON response. "
            f"HTTP status: {response.status_code}. "
            f"Response: {response.text[:2000]}"
        ) from exc

    if not isinstance(
        payload,
        dict,
    ):
        raise RuntimeError(
            "QuickBooks returned JSON, but the response root "
            "was not an object."
        )

    return payload


def extract_qbo_fault(
    response_payload: Dict[str, Any],
) -> Optional[str]:
    """Return a concise QuickBooks Fault description when present."""
    fault = response_payload.get(
        "Fault"
    )

    if not isinstance(
        fault,
        dict,
    ):
        return None

    errors = fault.get(
        "Error",
        []
    )

    if isinstance(
        errors,
        dict,
    ):
        errors = [errors]

    fault_parts: List[str] = []

    for error_item in errors:
        if not isinstance(
            error_item,
            dict,
        ):
            continue

        code = str(
            error_item.get(
                "code",
                ""
            )
        ).strip()

        message = str(
            error_item.get(
                "Message",
                ""
            )
        ).strip()

        detail = str(
            error_item.get(
                "Detail",
                ""
            )
        ).strip()

        combined_part = " | ".join(
            value
            for value in [
                code,
                message,
                detail,
            ]
            if value
        )

        if combined_part:
            fault_parts.append(
                combined_part
            )

    if fault_parts:
        return "; ".join(
            fault_parts
        )[:4000]

    return json.dumps(
        fault,
        default=str,
    )[:4000]


def sanitise_path_component(
    value: Any,
) -> str:
    """Create a safe lower-case OneLake folder-name component."""
    text_value = str(
        value
    ).strip().lower()

    text_value = SAFE_PATH_COMPONENT_PATTERN.sub(
        "_",
        text_value,
    ).strip("_")

    if not text_value:
        raise ValueError(
            "A raw-file path component resolved to a blank value."
        )

    return text_value


def calculate_cdc_retry_delay(
    attempt_number: int,
    retry_after_header: Optional[str] = None,
) -> int:
    """
    Honour a numeric Retry-After header or use capped exponential backoff.
    """
    if (
        retry_after_header
        and str(retry_after_header).strip().isdigit()
    ):
        return min(
            int(
                str(retry_after_header).strip()
            ),
            CDC_MAXIMUM_RETRY_DELAY_SECONDS,
        )

    calculated_delay = (
        CDC_INITIAL_RETRY_DELAY_SECONDS
        * (2 ** (attempt_number - 1))
    )

    return min(
        calculated_delay,
        CDC_MAXIMUM_RETRY_DELAY_SECONDS,
    )


def save_cdc_raw_response(
    source_system: str,
    source_company: str,
    source_object: str,
    request_sequence: int,
    response_payload: Dict[str, Any],
    request_parameters: Dict[str, str],
    http_status: int,
) -> str:
    """
    Preserve the complete source response with ingestion metadata.

    Raw data is intentionally retained before parsing or transformation.
    """
    ingested_utc = cdc_utc_now()

    safe_source_system = sanitise_path_component(
        source_system
    )

    safe_source_company = sanitise_path_component(
        source_company
    )

    safe_entity_name = sanitise_path_component(
        source_object
    )

    ingestion_date = ingested_utc.strftime(
        "%Y-%m-%d"
    )

    raw_path = (
        f"Files/{safe_source_system}/"
        f"{safe_source_company}/cdc_raw/"
        f"{safe_entity_name}/"
        f"ingestion_date={ingestion_date}/"
        f"run_id={PIPELINE_RUN_ID}/"
        f"request_{int(request_sequence):04d}.json"
    )

    raw_document = {
        "_ingestion_metadata": {
            "pipeline_run_id": str(
                PIPELINE_RUN_ID
            ),
            "source_system": source_system,
            "source_company": source_company,
            "source_object": source_object,
            "request_sequence": int(
                request_sequence
            ),
            "http_status": int(
                http_status
            ),
            "request_parameters": (
                request_parameters
            ),
            "ingested_utc": (
                ingested_utc.isoformat()
            ),
        },
        "payload": response_payload,
    }

    notebookutils.fs.put(
        raw_path,
        json.dumps(
            raw_document,
            ensure_ascii=False,
            indent=2,
            default=str,
        ),
        overwrite=True,
    )

    return raw_path


def extract_cdc_entity_records(
    payload: Dict[str, Any],
    source_object: str,
) -> List[Dict[str, Any]]:
    """
    Extract the changed-record array for one entity from a QuickBooks
    CDC response.
    """
    cdc_response = payload.get(
        "CDCResponse",
        []
    )

    if cdc_response is None:
        return []

    if isinstance(
        cdc_response,
        dict,
    ):
        cdc_response = [
            cdc_response
        ]

    if not isinstance(
        cdc_response,
        list,
    ):
        raise RuntimeError(
            f"CDCResponse for {source_object} is not a list or object."
        )

    changed_records: List[
        Dict[str, Any]
    ] = []

    for response_block in cdc_response:
        if not isinstance(
            response_block,
            dict,
        ):
            continue

        query_responses = response_block.get(
            "QueryResponse",
            []
        )

        if isinstance(
            query_responses,
            dict,
        ):
            query_responses = [
                query_responses
            ]

        if not isinstance(
            query_responses,
            list,
        ):
            raise RuntimeError(
                f"QueryResponse for {source_object} is invalid."
            )

        for query_response in query_responses:
            if not isinstance(
                query_response,
                dict,
            ):
                continue

            records = query_response.get(
                source_object,
                []
            )

            if records is None:
                continue

            if isinstance(
                records,
                dict,
            ):
                records = [
                    records
                ]

            if not isinstance(
                records,
                list,
            ):
                raise RuntimeError(
                    f"Entity collection {source_object} is not "
                    "a list or object."
                )

            for record in records:
                if not isinstance(
                    record,
                    dict,
                ):
                    raise RuntimeError(
                        f"{source_object} returned a record that "
                        "was not a JSON object."
                    )

                changed_records.append(
                    record
                )

    return changed_records


def split_active_and_deleted_records(
    changed_records: List[Dict[str, Any]],
) -> Tuple[
    List[Dict[str, Any]],
    List[Dict[str, Any]],
]:
    """Separate active changed records from QuickBooks deletion markers."""
    active_records: List[
        Dict[str, Any]
    ] = []

    deleted_records: List[
        Dict[str, Any]
    ] = []

    for record in changed_records:
        record_status = str(
            record.get(
                "status",
                record.get(
                    "Status",
                    "",
                ),
            )
        ).strip().lower()

        if record_status == "deleted":
            deleted_records.append(
                record
            )
        else:
            active_records.append(
                record
            )

    return (
        active_records,
        deleted_records,
    )


def execute_cdc_http_request(
    endpoint: str,
    request_parameters: Dict[str, str],
    source_object: str,
) -> Tuple[
    requests.Response,
    Dict[str, Any],
    int,
]:
    """
    Execute one CDC HTTP request with retry handling.

    Returns:
    - HTTP response
    - parsed response payload
    - final attempt number
    """
    if not str(endpoint).strip():
        raise ValueError(
            f"CDC endpoint is blank for {source_object}."
        )

    for attempt_number in range(
        1,
        CDC_MAXIMUM_ATTEMPTS + 1,
    ):
        try:
            response = requests.get(
                endpoint,
                headers=QBO_REQUEST_HEADERS,
                params=request_parameters,
                timeout=CDC_REQUEST_TIMEOUT_SECONDS,
            )

        except requests.RequestException as exc:
            if (
                attempt_number
                >= CDC_MAXIMUM_ATTEMPTS
            ):
                raise RuntimeError(
                    f"{source_object}: network request failed after "
                    f"{CDC_MAXIMUM_ATTEMPTS} attempts."
                ) from exc

            delay_seconds = calculate_cdc_retry_delay(
                attempt_number=attempt_number,
            )

            print(
                f"{source_object}: network error on attempt "
                f"{attempt_number}. Retrying in "
                f"{delay_seconds} seconds."
            )

            time.sleep(
                delay_seconds
            )
            continue


        if (
            response.status_code
            == CDC_SUCCESS_HTTP_STATUS_CODE
        ):
            response_payload = safe_response_payload(
                response
            )

            fault_text = extract_qbo_fault(
                response_payload
            )

            if fault_text:
                raise RuntimeError(
                    f"{source_object}: QuickBooks returned a Fault "
                    f"inside an HTTP 200 response: {fault_text}"
                )

            return (
                response,
                response_payload,
                attempt_number,
            )


        if (
            response.status_code
            not in CDC_TRANSIENT_HTTP_STATUS_CODES
        ):
            try:
                error_payload = response.json()
                qbo_fault = extract_qbo_fault(
                    error_payload
                )
            except ValueError:
                qbo_fault = None

            error_detail = (
                qbo_fault
                or response.text[:2000]
            )

            raise RuntimeError(
                f"{source_object}: QuickBooks CDC request returned "
                f"a non-retryable HTTP status "
                f"{response.status_code}. Details: {error_detail}"
            )


        if (
            attempt_number
            >= CDC_MAXIMUM_ATTEMPTS
        ):
            raise RuntimeError(
                f"{source_object}: QuickBooks CDC request failed after "
                f"{CDC_MAXIMUM_ATTEMPTS} attempts. Final HTTP status: "
                f"{response.status_code}. Response: "
                f"{response.text[:2000]}"
            )


        delay_seconds = calculate_cdc_retry_delay(
            attempt_number=attempt_number,
            retry_after_header=response.headers.get(
                "Retry-After"
            ),
        )

        print(
            f"{source_object}: transient HTTP "
            f"{response.status_code} on attempt "
            f"{attempt_number}. Retrying in "
            f"{delay_seconds} seconds."
        )

        time.sleep(
            delay_seconds
        )


    raise RuntimeError(
        f"{source_object}: CDC request processing ended unexpectedly."
    )


# =============================================================================
# 3. VALIDATE MANIFEST RECORDS BEFORE EXECUTION
# =============================================================================

manifest_errors: List[str] = []

for manifest_position, request_item in enumerate(
    CDC_REQUEST_MANIFEST,
    start=1,
):
    if not isinstance(
        request_item,
        dict,
    ):
        manifest_errors.append(
            f"Manifest item {manifest_position} is not a dictionary."
        )
        continue

    required_manifest_fields = [
        "request_sequence",
        "ingestion_id",
        "source_system",
        "source_company",
        "source_object",
        "cdc_endpoint",
        "changed_since_parameter",
        "minor_version",
        "request_status",
    ]

    missing_manifest_fields = [
        field_name
        for field_name in required_manifest_fields
        if field_name not in request_item
        or request_item[field_name] is None
        or not str(
            request_item[field_name]
        ).strip()
    ]

    if missing_manifest_fields:
        manifest_errors.append(
            f"Manifest item {manifest_position} is missing values: "
            + ", ".join(
                missing_manifest_fields
            )
        )
        continue

    if (
        str(
            request_item[
                "request_status"
            ]
        ).strip().upper()
        != CDC_READY_STATUS
    ):
        manifest_errors.append(
            f"Manifest item {manifest_position} is not READY."
        )


if manifest_errors:
    raise RuntimeError(
        "CDC execution manifest validation failed:\n"
        + "\n".join(
            f"{number}. {message}"
            for number, message in enumerate(
                manifest_errors,
                start=1,
            )
        )
    )


# =============================================================================
# 4. EXECUTE CDC REQUESTS
# =============================================================================

cdc_execution_results: List[
    Dict[str, Any]
] = []

CDC_CHANGED_RECORDS: Dict[
    str,
    List[Dict[str, Any]],
] = {}

CDC_ACTIVE_RECORDS: Dict[
    str,
    List[Dict[str, Any]],
] = {}

CDC_DELETED_RECORDS: Dict[
    str,
    List[Dict[str, Any]],
] = {}

CDC_EXECUTION_ERRORS: List[
    str
] = []


for request_item in sorted(
    CDC_REQUEST_MANIFEST,
    key=lambda item: int(
        item["request_sequence"]
    ),
):
    request_sequence = int(
        request_item[
            "request_sequence"
        ]
    )

    ingestion_id = int(
        request_item[
            "ingestion_id"
        ]
    )

    source_system = str(
        request_item[
            "source_system"
        ]
    ).strip().upper()

    source_company = str(
        request_item[
            "source_company"
        ]
    ).strip().upper()

    source_object = str(
        request_item[
            "source_object"
        ]
    ).strip()

    endpoint = str(
        request_item[
            "cdc_endpoint"
        ]
    ).strip()

    request_parameters = {
        "entities": source_object,
        "changedSince": str(
            request_item[
                "changed_since_parameter"
            ]
        ).strip(),
        "minorversion": str(
            request_item[
                "minor_version"
            ]
        ).strip(),
    }

    request_started_utc = cdc_utc_now()

    print(
        f"Executing CDC request "
        f"{request_sequence}: {source_object}"
    )

    response: Optional[
        requests.Response
    ] = None

    attempt_number: Optional[
        int
    ] = None

    try:
        (
            response,
            response_payload,
            attempt_number,
        ) = execute_cdc_http_request(
            endpoint=endpoint,
            request_parameters=request_parameters,
            source_object=source_object,
        )

        raw_path = save_cdc_raw_response(
            source_system=source_system,
            source_company=source_company,
            source_object=source_object,
            request_sequence=request_sequence,
            response_payload=response_payload,
            request_parameters=request_parameters,
            http_status=response.status_code,
        )

        changed_records = (
            extract_cdc_entity_records(
                payload=response_payload,
                source_object=source_object,
            )
        )

        (
            active_records,
            deleted_records,
        ) = split_active_and_deleted_records(
            changed_records
        )

        changed_count = len(
            changed_records
        )

        active_changed_count = len(
            active_records
        )

        deleted_count = len(
            deleted_records
        )

        if (
            changed_count
            != active_changed_count
            + deleted_count
        ):
            raise RuntimeError(
                f"{source_object}: changed-record reconciliation failed."
            )


        CDC_CHANGED_RECORDS[
            source_object
        ] = changed_records

        CDC_ACTIVE_RECORDS[
            source_object
        ] = active_records

        CDC_DELETED_RECORDS[
            source_object
        ] = deleted_records


        request_completed_utc = cdc_utc_now()

        cdc_execution_results.append(
            {
                "pipeline_run_id": str(
                    PIPELINE_RUN_ID
                ),
                "request_sequence": (
                    request_sequence
                ),
                "ingestion_id": (
                    ingestion_id
                ),
                "source_system": (
                    source_system
                ),
                "source_company": (
                    source_company
                ),
                "source_object": (
                    source_object
                ),
                "http_status": int(
                    response.status_code
                ),
                "attempt_count": int(
                    attempt_number
                ),
                "changed_records": int(
                    changed_count
                ),
                "active_changed_records": int(
                    active_changed_count
                ),
                "deleted_records": int(
                    deleted_count
                ),
                "raw_path": (
                    raw_path
                ),
                "request_started_utc": (
                    spark_compatible_utc(
                        request_started_utc
                    )
                ),
                "request_completed_utc": (
                    spark_compatible_utc(
                        request_completed_utc
                    )
                ),
                "duration_seconds": int(
                    max(
                        (
                            request_completed_utc
                            - request_started_utc
                        ).total_seconds(),
                        0,
                    )
                ),
                "status": "SUCCEEDED",
                "error_message": None,
            }
        )

        print(
            f"{source_object}: "
            f"{changed_count} changed, "
            f"{active_changed_count} active, "
            f"{deleted_count} deleted"
        )


    except Exception as exc:
        error_text = str(
            exc
        )[:4000]

        request_completed_utc = cdc_utc_now()

        CDC_EXECUTION_ERRORS.append(
            f"{source_object}: {error_text}"
        )

        # Ensure downstream dictionaries still contain the entity.
        CDC_CHANGED_RECORDS[
            source_object
        ] = []

        CDC_ACTIVE_RECORDS[
            source_object
        ] = []

        CDC_DELETED_RECORDS[
            source_object
        ] = []


        cdc_execution_results.append(
            {
                "pipeline_run_id": str(
                    PIPELINE_RUN_ID
                ),
                "request_sequence": (
                    request_sequence
                ),
                "ingestion_id": (
                    ingestion_id
                ),
                "source_system": (
                    source_system
                ),
                "source_company": (
                    source_company
                ),
                "source_object": (
                    source_object
                ),
                "http_status": (
                    int(
                        response.status_code
                    )
                    if response is not None
                    else None
                ),
                "attempt_count": (
                    int(attempt_number)
                    if attempt_number is not None
                    else CDC_MAXIMUM_ATTEMPTS
                ),
                "changed_records": 0,
                "active_changed_records": 0,
                "deleted_records": 0,
                "raw_path": None,
                "request_started_utc": (
                    spark_compatible_utc(
                        request_started_utc
                    )
                ),
                "request_completed_utc": (
                    spark_compatible_utc(
                        request_completed_utc
                    )
                ),
                "duration_seconds": int(
                    max(
                        (
                            request_completed_utc
                            - request_started_utc
                        ).total_seconds(),
                        0,
                    )
                ),
                "status": "FAILED",
                "error_message": (
                    error_text
                ),
            }
        )

        print(
            f"{source_object}: FAILED — "
            f"{error_text}"
        )


# =============================================================================
# 5. CREATE THE EXECUTION RESULTS DATAFRAME
# =============================================================================

cdc_execution_schema = StructType([
    StructField(
        "pipeline_run_id",
        StringType(),
        False,
    ),
    StructField(
        "request_sequence",
        IntegerType(),
        False,
    ),
    StructField(
        "ingestion_id",
        IntegerType(),
        False,
    ),
    StructField(
        "source_system",
        StringType(),
        False,
    ),
    StructField(
        "source_company",
        StringType(),
        False,
    ),
    StructField(
        "source_object",
        StringType(),
        False,
    ),
    StructField(
        "http_status",
        IntegerType(),
        True,
    ),
    StructField(
        "attempt_count",
        IntegerType(),
        False,
    ),
    StructField(
        "changed_records",
        LongType(),
        False,
    ),
    StructField(
        "active_changed_records",
        LongType(),
        False,
    ),
    StructField(
        "deleted_records",
        LongType(),
        False,
    ),
    StructField(
        "raw_path",
        StringType(),
        True,
    ),
    StructField(
        "request_started_utc",
        TimestampType(),
        False,
    ),
    StructField(
        "request_completed_utc",
        TimestampType(),
        False,
    ),
    StructField(
        "duration_seconds",
        LongType(),
        False,
    ),
    StructField(
        "status",
        StringType(),
        False,
    ),
    StructField(
        "error_message",
        StringType(),
        True,
    ),
])


CDC_EXECUTION_RESULTS_DF = (
    spark.createDataFrame(
        cdc_execution_results,
        schema=cdc_execution_schema,
    )
)


# =============================================================================
# 6. VALIDATE EXECUTION RESULTS
# =============================================================================

request_count = (
    CDC_EXECUTION_RESULTS_DF.count()
)

expected_request_count = len(
    CDC_REQUEST_MANIFEST
)


if request_count != expected_request_count:
    raise RuntimeError(
        "CDC execution-result count does not match the request manifest. "
        f"Expected {expected_request_count}; created {request_count}."
    )


duplicate_execution_count = (
    CDC_EXECUTION_RESULTS_DF
    .groupBy(
        "pipeline_run_id",
        "request_sequence",
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)


if duplicate_execution_count > 0:
    raise RuntimeError(
        "The CDC execution results contain duplicate request sequences."
    )


successful_request_count = (
    CDC_EXECUTION_RESULTS_DF
    .filter(
        F.col("status")
        == "SUCCEEDED"
    )
    .count()
)


failed_request_count = (
    CDC_EXECUTION_RESULTS_DF
    .filter(
        F.col("status")
        == "FAILED"
    )
    .count()
)


status_reconciliation_count = (
    successful_request_count
    + failed_request_count
)

if (
    status_reconciliation_count
    != request_count
):
    raise RuntimeError(
        "CDC execution status reconciliation failed."
    )


aggregate_metrics = (
    CDC_EXECUTION_RESULTS_DF
    .agg(
        F.coalesce(
            F.sum(
                "changed_records"
            ),
            F.lit(0),
        ).alias(
            "total_changed_records"
        ),
        F.coalesce(
            F.sum(
                "active_changed_records"
            ),
            F.lit(0),
        ).alias(
            "total_active_changed_records"
        ),
        F.coalesce(
            F.sum(
                "deleted_records"
            ),
            F.lit(0),
        ).alias(
            "total_deleted_records"
        ),
        F.coalesce(
            F.sum(
                "duration_seconds"
            ),
            F.lit(0),
        ).alias(
            "total_request_duration_seconds"
        ),
    )
    .first()
)


TOTAL_CHANGED_RECORDS = int(
    aggregate_metrics[
        "total_changed_records"
    ]
)

TOTAL_ACTIVE_CHANGED_RECORDS = int(
    aggregate_metrics[
        "total_active_changed_records"
    ]
)

TOTAL_DELETED_RECORDS = int(
    aggregate_metrics[
        "total_deleted_records"
    ]
)

TOTAL_REQUEST_DURATION_SECONDS = int(
    aggregate_metrics[
        "total_request_duration_seconds"
    ]
)


if (
    TOTAL_CHANGED_RECORDS
    != TOTAL_ACTIVE_CHANGED_RECORDS
    + TOTAL_DELETED_RECORDS
):
    raise RuntimeError(
        "Aggregate CDC changed-record reconciliation failed."
    )


# =============================================================================
# 7. DISPLAY THE SAFE EXECUTION RESULTS
# =============================================================================

display(
    CDC_EXECUTION_RESULTS_DF
    .select(
        "request_sequence",
        "ingestion_id",
        "source_object",
        "http_status",
        "attempt_count",
        "changed_records",
        "active_changed_records",
        "deleted_records",
        "duration_seconds",
        "status",
        "error_message",
    )
    .orderBy(
        "request_sequence"
    )
)


# =============================================================================
# 8. PUBLISH DOWNSTREAM EXECUTION STATE
# =============================================================================

CDC_EXTRACTION_SUCCEEDED = (
    failed_request_count == 0
)

CDC_EXTRACTION_COMPLETED_WITH_ERRORS = (
    failed_request_count > 0
)

CDC_SUCCESSFUL_ENTITIES = [
    row["source_object"]
    for row in (
        CDC_EXECUTION_RESULTS_DF
        .filter(
            F.col("status")
            == "SUCCEEDED"
        )
        .select(
            "source_object"
        )
        .orderBy(
            "source_object"
        )
        .collect()
    )
]

CDC_FAILED_ENTITIES = [
    row["source_object"]
    for row in (
        CDC_EXECUTION_RESULTS_DF
        .filter(
            F.col("status")
            == "FAILED"
        )
        .select(
            "source_object"
        )
        .orderBy(
            "source_object"
        )
        .collect()
    )
]


# =============================================================================
# 9. SAFE EXECUTION SUMMARY
# =============================================================================

print("=" * 80)
print("QUICKBOOKS CDC EXECUTION SUMMARY")
print("=" * 80)
print(f"Pipeline run ID               : {PIPELINE_RUN_ID}")
print(f"Requests expected             : {expected_request_count}")
print(f"Requests executed             : {request_count}")
print(f"Requests succeeded            : {successful_request_count}")
print(f"Requests failed               : {failed_request_count}")
print(f"Total changed records         : {TOTAL_CHANGED_RECORDS}")
print(
    "Total active changed records  : "
    f"{TOTAL_ACTIVE_CHANGED_RECORDS}"
)
print(f"Total deleted records         : {TOTAL_DELETED_RECORDS}")
print(
    "Total request duration seconds: "
    f"{TOTAL_REQUEST_DURATION_SECONDS}"
)
print("=" * 80)


if CDC_EXECUTION_ERRORS:
    print(
        "CDC extraction completed with one or more entity errors."
    )

    if FAIL_ON_ENTITY_ERROR:
        raise RuntimeError(
            "One or more QuickBooks CDC requests failed:\n"
            + "\n".join(
                f"{number}. {message}"
                for number, message in enumerate(
                    CDC_EXECUTION_ERRORS,
                    start=1,
                )
            )
        )

    print(
        "FAIL_ON_ENTITY_ERROR is False, so successful entities "
        "may continue to downstream processing."
    )
    print(
        "QuickBooks CDC extraction: COMPLETED_WITH_ERRORS"
    )

else:
    print(
        "QuickBooks CDC extraction: SUCCEEDED"
    )

StatementMeta(, 512b264d-925e-4515-a439-f76b522e9d96, -1, Cancelled, , Cancelled, True)

In [28]:
# =============================================================================
# CELL 7 — PERSIST CDC CHANGE LOG AND MERGE INTO BRONZE
# QuickBooks Online Incremental Bronze Ingestion
#
# Purpose:
# 1. Preserve every QuickBooks CDC event in an immutable Delta change log.
# 2. Upsert active records into current-state Bronze header tables.
# 3. Replace transaction lines for changed parent records.
# 4. Remove deleted records from current-state Bronze.
# 5. Preserve deletion tombstones in the CDC change log and raw JSON.
# 6. Record entity-level merge metrics for audit and validation.
#
# Prerequisite:
# Cell 6 must publish:
# - CDC_CHANGED_RECORDS
# - CDC_ACTIVE_RECORDS
# - CDC_DELETED_RECORDS
# - CDC_EXECUTION_RESULTS_DF
#
# Idempotency:
# Reprocessing the same pipeline run does not create duplicate CDC log
# or merge-audit records.
# =============================================================================


# -----------------------------------------------------------------------------
# Imports
# -----------------------------------------------------------------------------

import hashlib
import json
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import (
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)


# -----------------------------------------------------------------------------
# Constants
# -----------------------------------------------------------------------------

CDC_CHANGE_LOG_TABLE = "qbo_es_cdc_change_log"
BRONZE_MERGE_AUDIT_TABLE = "qbo_es_incremental_merge_audit"

CELL_7_STARTED_UTC = datetime.now(
    timezone.utc
)


# =============================================================================
# 1. VALIDATE REQUIRED UPSTREAM OBJECTS
# =============================================================================

required_objects = [
    "PIPELINE_RUN_ID",
    "SOURCE_SYSTEM",
    "SOURCE_COMPANY",
    "SOURCE_ENVIRONMENT",
    "QBO_REALM_ID",
    "QBO_MINOR_VERSION",
    "CDC_CHANGED_RECORDS",
    "CDC_ACTIVE_RECORDS",
    "CDC_DELETED_RECORDS",
    "CDC_EXECUTION_RESULTS_DF",
    "INCREMENTAL_ENTITIES",
    "WATERMARK_RUNTIME_DF",
    "FAIL_ON_ENTITY_ERROR",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Cell 7 cannot run because these upstream objects are missing: "
        + ", ".join(missing_objects)
        + ". Run Cells 1 through 6 first."
    )


if not isinstance(
    CDC_CHANGED_RECORDS,
    dict,
):
    raise TypeError(
        "CDC_CHANGED_RECORDS must be a Python dictionary."
    )


if not isinstance(
    CDC_ACTIVE_RECORDS,
    dict,
):
    raise TypeError(
        "CDC_ACTIVE_RECORDS must be a Python dictionary."
    )


if not isinstance(
    CDC_DELETED_RECORDS,
    dict,
):
    raise TypeError(
        "CDC_DELETED_RECORDS must be a Python dictionary."
    )


if not isinstance(
    INCREMENTAL_ENTITIES,
    list,
):
    raise TypeError(
        "INCREMENTAL_ENTITIES must be a Python list."
    )


if not INCREMENTAL_ENTITIES:
    raise RuntimeError(
        "INCREMENTAL_ENTITIES is empty."
    )


# Do not continue downstream with failed extraction entities.
if "CDC_FAILED_ENTITIES" in globals():
    failed_extraction_entities = list(
        CDC_FAILED_ENTITIES
    )

    if (
        failed_extraction_entities
        and FAIL_ON_ENTITY_ERROR
    ):
        raise RuntimeError(
            "Cell 7 cannot continue because CDC extraction failed for: "
            + ", ".join(
                failed_extraction_entities
            )
        )


# =============================================================================
# 2. BUILD RUNTIME LOOKUPS
# =============================================================================

entity_metadata_lookup = {
    str(
        entity["source_object"]
    ).strip(): entity
    for entity in INCREMENTAL_ENTITIES
}


if len(entity_metadata_lookup) != len(
    INCREMENTAL_ENTITIES
):
    raise RuntimeError(
        "Duplicate source objects exist in INCREMENTAL_ENTITIES."
    )


watermark_runtime_rows = (
    WATERMARK_RUNTIME_DF
    .collect()
)


watermark_lookup = {
    str(
        row["source_object"]
    ).strip(): row.asDict(
        recursive=True
    )
    for row in watermark_runtime_rows
}


if len(watermark_lookup) != len(
    watermark_runtime_rows
):
    raise RuntimeError(
        "Duplicate source objects exist in WATERMARK_RUNTIME_DF."
    )


# =============================================================================
# 3. HELPER FUNCTIONS
# =============================================================================

def cell7_utc_now() -> datetime:
    """Return the current timezone-aware UTC timestamp."""
    return datetime.now(
        timezone.utc
    )


def spark_compatible_utc(
    value: Optional[datetime],
) -> Optional[datetime]:
    """
    Convert a datetime into a timezone-naive UTC value suitable for
    Spark TimestampType DataFrame construction.
    """
    if value is None:
        return None

    if not isinstance(
        value,
        datetime,
    ):
        raise TypeError(
            "Spark timestamp values must be datetime objects or None. "
            f"Received: {type(value).__name__}."
        )

    if value.tzinfo is None:
        return value

    return (
        value
        .astimezone(timezone.utc)
        .replace(tzinfo=None)
    )


def serialise_json(
    value: Any,
) -> str:
    """Return stable compact JSON for hashing and storage."""
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    )


def calculate_payload_hash(
    value: Any,
) -> str:
    """Calculate a SHA-256 hash of the canonical JSON payload."""
    encoded_payload = serialise_json(
        value
    ).encode(
        "utf-8"
    )

    return hashlib.sha256(
        encoded_payload
    ).hexdigest()


def nullable_string(
    value: Any,
) -> Optional[str]:
    """Return a trimmed string or None."""
    if value is None:
        return None

    text_value = str(
        value
    ).strip()

    if not text_value:
        return None

    return text_value


def extract_reference_value(
    record: Dict[str, Any],
    reference_name: str,
) -> Optional[str]:
    """Extract the value field from a QuickBooks reference object."""
    reference_value = record.get(
        reference_name
    )

    if isinstance(
        reference_value,
        dict,
    ):
        return nullable_string(
            reference_value.get(
                "value"
            )
        )

    return nullable_string(
        reference_value
    )


def get_record_status(
    record: Dict[str, Any],
) -> Optional[str]:
    """Read either the status or Status field."""
    return nullable_string(
        record.get(
            "status",
            record.get(
                "Status"
            ),
        )
    )


def is_deleted_record(
    record: Dict[str, Any],
) -> bool:
    """Identify a QuickBooks CDC deletion tombstone."""
    return (
        str(
            get_record_status(
                record
            )
            or ""
        )
        .strip()
        .lower()
        == "deleted"
    )


def get_source_metadata(
    record: Dict[str, Any],
) -> Dict[str, Any]:
    """Return the QuickBooks MetaData object safely."""
    metadata = record.get(
        "MetaData",
        {},
    )

    if metadata is None:
        return {}

    if not isinstance(
        metadata,
        dict,
    ):
        return {}

    return metadata


def get_source_last_updated_time(
    record: Dict[str, Any],
) -> Optional[str]:
    """Read QuickBooks MetaData.LastUpdatedTime."""
    return nullable_string(
        get_source_metadata(
            record
        ).get(
            "LastUpdatedTime"
        )
    )


def get_source_created_time(
    record: Dict[str, Any],
) -> Optional[str]:
    """Read QuickBooks MetaData.CreateTime."""
    return nullable_string(
        get_source_metadata(
            record
        ).get(
            "CreateTime"
        )
    )


def get_source_record_id(
    record: Dict[str, Any],
) -> str:
    """Return the mandatory QuickBooks source record ID."""
    source_record_id = nullable_string(
        record.get(
            "Id"
        )
    )

    if not source_record_id:
        raise ValueError(
            "A CDC record does not contain a valid Id."
        )

    return source_record_id


def record_sort_key(
    record: Dict[str, Any],
) -> Tuple[str, str]:
    """
    Provide a deterministic ordering key for duplicate IDs within one
    CDC batch.
    """
    return (
        get_source_last_updated_time(
            record
        )
        or "",
        calculate_payload_hash(
            record
        ),
    )


def deduplicate_cdc_records(
    source_object: str,
    records: Any,
) -> List[Dict[str, Any]]:
    """
    Keep one deterministic latest record per QuickBooks source ID
    within the current CDC batch.
    """
    if records is None:
        return []

    if isinstance(
        records,
        dict,
    ):
        records = [
            records
        ]

    if not isinstance(
        records,
        list,
    ):
        raise TypeError(
            f"{source_object}: changed records must be a list."
        )

    records_by_id: Dict[
        str,
        Dict[str, Any],
    ] = {}

    for record in records:
        if not isinstance(
            record,
            dict,
        ):
            raise TypeError(
                f"{source_object}: every CDC record must be a dictionary."
            )

        source_record_id = get_source_record_id(
            record
        )

        existing_record = records_by_id.get(
            source_record_id
        )

        if existing_record is None:
            records_by_id[
                source_record_id
            ] = record
            continue

        if (
            record_sort_key(
                record
            )
            >= record_sort_key(
                existing_record
            )
        ):
            records_by_id[
                source_record_id
            ] = record

    return list(
        records_by_id.values()
    )


def build_header_row(
    record: Dict[str, Any],
    source_object: str,
    ingested_utc: datetime,
) -> Dict[str, Any]:
    """Build one standardised Bronze current-state header row."""
    return {
        "source_record_id": get_source_record_id(
            record
        ),
        "sync_token": nullable_string(
            record.get(
                "SyncToken"
            )
        ),
        "source_created_time": get_source_created_time(
            record
        ),
        "source_last_updated_time":
            get_source_last_updated_time(
                record
            ),
        "document_number": nullable_string(
            record.get(
                "DocNumber"
            )
            or record.get(
                "Name"
            )
            or record.get(
                "DisplayName"
            )
            or record.get(
                "CompanyName"
            )
        ),
        "transaction_date": nullable_string(
            record.get(
                "TxnDate"
            )
        ),
        "currency_code": extract_reference_value(
            record,
            "CurrencyRef",
        ),
        "total_amount": nullable_string(
            record.get(
                "TotalAmt"
            )
        ),
        "balance_amount": nullable_string(
            record.get(
                "Balance"
            )
        ),
        "payload_json": serialise_json(
            record
        ),
        "payload_hash": calculate_payload_hash(
            record
        ),
        "_source_system": str(
            SOURCE_SYSTEM
        ),
        "_source_company": str(
            SOURCE_COMPANY
        ),
        "_source_environment": str(
            SOURCE_ENVIRONMENT
        ),
        "_source_entity": str(
            source_object
        ),
        "_realm_id": str(
            QBO_REALM_ID
        ),
        "_pipeline_run_id": str(
            PIPELINE_RUN_ID
        ),
        "_ingested_utc": spark_compatible_utc(
            ingested_utc
        ),
        "_api_minor_version": str(
            QBO_MINOR_VERSION
        ),
    }


def build_line_rows(
    record: Dict[str, Any],
    source_object: str,
    ingested_utc: datetime,
) -> List[Dict[str, Any]]:
    """Build current-state Bronze transaction-line rows."""
    parent_record_id = get_source_record_id(
        record
    )

    source_lines = (
        record.get(
            "Line",
            [],
        )
        or []
    )

    if isinstance(
        source_lines,
        dict,
    ):
        source_lines = [
            source_lines
        ]

    if not isinstance(
        source_lines,
        list,
    ):
        raise TypeError(
            f"{source_object} record {parent_record_id} contains an "
            "invalid Line collection."
        )

    line_rows: List[
        Dict[str, Any]
    ] = []

    for line_number, line in enumerate(
        source_lines,
        start=1,
    ):
        if not isinstance(
            line,
            dict,
        ):
            raise TypeError(
                f"{source_object} record {parent_record_id} contains "
                "a line that is not a JSON object."
            )

        line_rows.append(
            {
                "parent_record_id": parent_record_id,
                "line_id": nullable_string(
                    line.get(
                        "Id"
                    )
                ),
                "line_number": int(
                    line_number
                ),
                "line_amount": nullable_string(
                    line.get(
                        "Amount"
                    )
                ),
                "line_description": nullable_string(
                    line.get(
                        "Description"
                    )
                ),
                "detail_type": nullable_string(
                    line.get(
                        "DetailType"
                    )
                ),
                "line_payload_json": serialise_json(
                    line
                ),
                "line_payload_hash": calculate_payload_hash(
                    line
                ),
                "_source_system": str(
                    SOURCE_SYSTEM
                ),
                "_source_company": str(
                    SOURCE_COMPANY
                ),
                "_source_environment": str(
                    SOURCE_ENVIRONMENT
                ),
                "_source_entity": str(
                    source_object
                ),
                "_realm_id": str(
                    QBO_REALM_ID
                ),
                "_pipeline_run_id": str(
                    PIPELINE_RUN_ID
                ),
                "_ingested_utc": spark_compatible_utc(
                    ingested_utc
                ),
            }
        )

    return line_rows


def upsert_idempotent_audit_table(
    source_df,
    table_name: str,
    merge_condition: str,
) -> None:
    """
    Insert or update a Delta audit table without duplicating the same
    pipeline-run records.
    """
    if spark.catalog.tableExists(
        table_name
    ):
        target_delta = DeltaTable.forName(
            spark,
            table_name,
        )

        (
            target_delta.alias(
                "target"
            )
            .merge(
                source_df.alias(
                    "source"
                ),
                merge_condition,
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

    else:
        (
            source_df
            .write
            .format(
                "delta"
            )
            .mode(
                "overwrite"
            )
            .saveAsTable(
                table_name
            )
        )


# =============================================================================
# 4. NORMALISE CDC RECORDS AND BUILD CHANGE-LOG ROWS
# =============================================================================

cdc_change_log_rows: List[
    Dict[str, Any]
] = []

NORMALISED_CDC_RECORDS: Dict[
    str,
    List[Dict[str, Any]],
] = {}

normalisation_errors: List[
    str
] = []


for entity in INCREMENTAL_ENTITIES:
    source_object = str(
        entity["source_object"]
    ).strip()

    entity_metadata = entity_metadata_lookup.get(
        source_object
    )

    watermark_runtime = watermark_lookup.get(
        source_object
    )

    if entity_metadata is None:
        normalisation_errors.append(
            f"{source_object}: entity metadata was not found."
        )
        continue

    if watermark_runtime is None:
        normalisation_errors.append(
            f"{source_object}: watermark runtime was not found."
        )
        continue

    try:
        raw_records = CDC_CHANGED_RECORDS.get(
            source_object,
            [],
        )

        normalised_records = deduplicate_cdc_records(
            source_object=source_object,
            records=raw_records,
        )

        NORMALISED_CDC_RECORDS[
            source_object
        ] = normalised_records

        for record in normalised_records:
            source_record_id = get_source_record_id(
                record
            )

            deleted_record = is_deleted_record(
                record
            )

            payload_json = serialise_json(
                record
            )

            payload_hash = calculate_payload_hash(
                record
            )

            cdc_change_log_rows.append(
                {
                    "pipeline_run_id": str(
                        PIPELINE_RUN_ID
                    ),
                    "ingestion_id": int(
                        entity_metadata[
                            "ingestion_id"
                        ]
                    ),
                    "source_system": str(
                        SOURCE_SYSTEM
                    ),
                    "source_company": str(
                        SOURCE_COMPANY
                    ),
                    "source_environment": str(
                        SOURCE_ENVIRONMENT
                    ),
                    "source_object": source_object,
                    "source_record_id": source_record_id,
                    "cdc_operation": (
                        "DELETE"
                        if deleted_record
                        else "UPSERT"
                    ),
                    "cdc_status": get_record_status(
                        record
                    ),
                    "source_last_updated_time":
                        get_source_last_updated_time(
                            record
                        ),
                    "payload_json": payload_json,
                    "payload_hash": payload_hash,
                    "previous_watermark_utc":
                        spark_compatible_utc(
                            watermark_runtime[
                                "previous_watermark_utc"
                            ]
                        ),
                    "extraction_start_utc":
                        spark_compatible_utc(
                            watermark_runtime[
                                "extraction_start_utc"
                            ]
                        ),
                    "extraction_end_utc":
                        spark_compatible_utc(
                            watermark_runtime[
                                "extraction_end_utc"
                            ]
                        ),
                    "ingested_utc":
                        spark_compatible_utc(
                            cell7_utc_now()
                        ),
                }
            )

    except Exception as exc:
        normalisation_errors.append(
            f"{source_object}: {exc}"
        )


if normalisation_errors:
    raise RuntimeError(
        "CDC record normalisation failed:\n"
        + "\n".join(
            f"{number}. {message}"
            for number, message in enumerate(
                normalisation_errors,
                start=1,
            )
        )
    )


# Ensure all configured entities are represented.
for entity in INCREMENTAL_ENTITIES:
    source_object = str(
        entity["source_object"]
    ).strip()

    NORMALISED_CDC_RECORDS.setdefault(
        source_object,
        [],
    )


# =============================================================================
# 5. CREATE AND PERSIST THE CDC CHANGE-LOG BATCH
# =============================================================================

cdc_change_log_schema = StructType([
    StructField(
        "pipeline_run_id",
        StringType(),
        False,
    ),
    StructField(
        "ingestion_id",
        IntegerType(),
        False,
    ),
    StructField(
        "source_system",
        StringType(),
        False,
    ),
    StructField(
        "source_company",
        StringType(),
        False,
    ),
    StructField(
        "source_environment",
        StringType(),
        False,
    ),
    StructField(
        "source_object",
        StringType(),
        False,
    ),
    StructField(
        "source_record_id",
        StringType(),
        False,
    ),
    StructField(
        "cdc_operation",
        StringType(),
        False,
    ),
    StructField(
        "cdc_status",
        StringType(),
        True,
    ),
    StructField(
        "source_last_updated_time",
        StringType(),
        True,
    ),
    StructField(
        "payload_json",
        StringType(),
        False,
    ),
    StructField(
        "payload_hash",
        StringType(),
        False,
    ),
    StructField(
        "previous_watermark_utc",
        TimestampType(),
        True,
    ),
    StructField(
        "extraction_start_utc",
        TimestampType(),
        False,
    ),
    StructField(
        "extraction_end_utc",
        TimestampType(),
        False,
    ),
    StructField(
        "ingested_utc",
        TimestampType(),
        False,
    ),
])


CDC_CHANGE_LOG_BATCH_DF = spark.createDataFrame(
    cdc_change_log_rows,
    schema=cdc_change_log_schema,
)


if cdc_change_log_rows:
    # The existing schema is preserved. The deterministic merge condition
    # prevents duplicate CDC events for the same retried pipeline run.
    upsert_idempotent_audit_table(
        source_df=CDC_CHANGE_LOG_BATCH_DF,
        table_name=CDC_CHANGE_LOG_TABLE,
        merge_condition="""
            target.pipeline_run_id = source.pipeline_run_id
            AND target.source_object = source.source_object
            AND target.source_record_id = source.source_record_id
            AND target.cdc_operation = source.cdc_operation
            AND target.payload_hash = source.payload_hash
        """,
    )


# =============================================================================
# 6. MERGE EVERY ENTITY INTO CURRENT-STATE BRONZE
# =============================================================================

bronze_merge_results: List[
    Dict[str, Any]
] = []

BRONZE_MERGE_ERRORS: List[
    str
] = []


for entity in INCREMENTAL_ENTITIES:
    source_object = str(
        entity["source_object"]
    ).strip()

    target_table = str(
        entity["target_table"]
    ).strip()

    target_line_table = entity.get(
        "target_line_table"
    )

    if target_line_table is not None:
        target_line_table = str(
            target_line_table
        ).strip() or None

    changed_records = NORMALISED_CDC_RECORDS.get(
        source_object,
        [],
    )

    active_records = [
        record
        for record in changed_records
        if not is_deleted_record(
            record
        )
    ]

    deleted_record_ids = sorted({
        get_source_record_id(
            record
        )
        for record in changed_records
        if is_deleted_record(
            record
        )
    })

    affected_parent_ids = sorted({
        get_source_record_id(
            record
        )
        for record in changed_records
    })

    entity_started_utc = cell7_utc_now()

    upserted_header_rows = 0
    deleted_header_rows = 0
    replaced_line_rows = 0
    deleted_existing_line_rows = 0

    try:
        if not spark.catalog.tableExists(
            target_table
        ):
            raise RuntimeError(
                f"Target Bronze table does not exist: {target_table}. "
                "Run the initial full-load framework first."
            )

        if (
            target_line_table
            and not spark.catalog.tableExists(
                target_line_table
            )
        ):
            raise RuntimeError(
                "The configured Bronze line table does not exist: "
                f"{target_line_table}."
            )

        target_header_df = spark.table(
            target_table
        )

        required_header_columns = {
            "source_record_id",
        }

        missing_header_columns = sorted(
            required_header_columns
            - set(
                target_header_df.columns
            )
        )

        if missing_header_columns:
            raise RuntimeError(
                f"{target_table} is missing required columns: "
                + ", ".join(
                    missing_header_columns
                )
            )

        target_delta_table = DeltaTable.forName(
            spark,
            target_table,
        )


        # ---------------------------------------------------------------------
        # Delete current-state header rows represented by CDC tombstones
        # ---------------------------------------------------------------------

        if deleted_record_ids:
            deleted_header_rows = (
                target_header_df
                .filter(
                    F.col(
                        "source_record_id"
                    ).isin(
                        deleted_record_ids
                    )
                )
                .count()
            )

            target_delta_table.delete(
                F.col(
                    "source_record_id"
                ).isin(
                    deleted_record_ids
                )
            )


        # ---------------------------------------------------------------------
        # Upsert active current-state header rows
        # ---------------------------------------------------------------------

        if active_records:
            entity_ingested_utc = cell7_utc_now()

            header_rows = [
                build_header_row(
                    record=record,
                    source_object=source_object,
                    ingested_utc=entity_ingested_utc,
                )
                for record in active_records
            ]

            target_header_schema = target_header_df.schema

            source_header_df = (
                spark.createDataFrame(
                    header_rows,
                    schema=target_header_schema,
                )
                .dropDuplicates([
                    "source_record_id"
                ])
            )

            upserted_header_rows = (
                source_header_df.count()
            )

            (
                target_delta_table
                .alias(
                    "target"
                )
                .merge(
                    source_header_df.alias(
                        "source"
                    ),
                    """
                    target.source_record_id =
                        source.source_record_id
                    """,
                )
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute()
            )


        # ---------------------------------------------------------------------
        # Replace current-state transaction lines for all affected parents
        # ---------------------------------------------------------------------

        if (
            target_line_table
            and affected_parent_ids
        ):
            target_line_df = spark.table(
                target_line_table
            )

            required_line_columns = {
                "parent_record_id",
                "line_number",
            }

            missing_line_columns = sorted(
                required_line_columns
                - set(
                    target_line_df.columns
                )
            )

            if missing_line_columns:
                raise RuntimeError(
                    f"{target_line_table} is missing required columns: "
                    + ", ".join(
                        missing_line_columns
                    )
                )

            target_line_delta = DeltaTable.forName(
                spark,
                target_line_table,
            )

            deleted_existing_line_rows = (
                target_line_df
                .filter(
                    F.col(
                        "parent_record_id"
                    ).isin(
                        affected_parent_ids
                    )
                )
                .count()
            )

            # Remove previous line versions for changed and deleted parents.
            target_line_delta.delete(
                F.col(
                    "parent_record_id"
                ).isin(
                    affected_parent_ids
                )
            )

            all_new_line_rows: List[
                Dict[str, Any]
            ] = []

            line_ingested_utc = cell7_utc_now()

            for record in active_records:
                all_new_line_rows.extend(
                    build_line_rows(
                        record=record,
                        source_object=source_object,
                        ingested_utc=line_ingested_utc,
                    )
                )

            if all_new_line_rows:
                target_line_schema = target_line_df.schema

                source_line_df = (
                    spark.createDataFrame(
                        all_new_line_rows,
                        schema=target_line_schema,
                    )
                    .dropDuplicates([
                        "parent_record_id",
                        "line_number",
                    ])
                )

                replaced_line_rows = (
                    source_line_df.count()
                )

                (
                    source_line_df
                    .write
                    .format(
                        "delta"
                    )
                    .mode(
                        "append"
                    )
                    .saveAsTable(
                        target_line_table
                    )
                )


        entity_completed_utc = cell7_utc_now()

        bronze_merge_results.append(
            {
                "pipeline_run_id": str(
                    PIPELINE_RUN_ID
                ),
                "ingestion_id": int(
                    entity["ingestion_id"]
                ),
                "source_object": source_object,
                "target_table": target_table,
                "target_line_table": target_line_table,
                "changed_records_received": int(
                    len(
                        changed_records
                    )
                ),
                "headers_upserted": int(
                    upserted_header_rows
                ),
                "headers_deleted": int(
                    deleted_header_rows
                ),
                "lines_replaced": int(
                    replaced_line_rows
                ),
                "started_utc": spark_compatible_utc(
                    entity_started_utc
                ),
                "completed_utc": spark_compatible_utc(
                    entity_completed_utc
                ),
                "status": "SUCCEEDED",
                "error_message": None,
            }
        )

        print(
            f"{source_object}: "
            f"{upserted_header_rows} headers upserted, "
            f"{deleted_header_rows} headers deleted, "
            f"{deleted_existing_line_rows} previous lines removed, "
            f"{replaced_line_rows} current lines written"
        )

    except Exception as exc:
        error_text = str(
            exc
        )[:4000]

        entity_completed_utc = cell7_utc_now()

        BRONZE_MERGE_ERRORS.append(
            f"{source_object}: {error_text}"
        )

        bronze_merge_results.append(
            {
                "pipeline_run_id": str(
                    PIPELINE_RUN_ID
                ),
                "ingestion_id": int(
                    entity["ingestion_id"]
                ),
                "source_object": source_object,
                "target_table": target_table,
                "target_line_table": target_line_table,
                "changed_records_received": int(
                    len(
                        changed_records
                    )
                ),
                "headers_upserted": 0,
                "headers_deleted": 0,
                "lines_replaced": 0,
                "started_utc": spark_compatible_utc(
                    entity_started_utc
                ),
                "completed_utc": spark_compatible_utc(
                    entity_completed_utc
                ),
                "status": "FAILED",
                "error_message": error_text,
            }
        )

        print(
            f"{source_object}: FAILED — {error_text}"
        )


# =============================================================================
# 7. CREATE THE BRONZE MERGE RESULTS DATAFRAME
# =============================================================================

bronze_merge_schema = StructType([
    StructField(
        "pipeline_run_id",
        StringType(),
        False,
    ),
    StructField(
        "ingestion_id",
        IntegerType(),
        False,
    ),
    StructField(
        "source_object",
        StringType(),
        False,
    ),
    StructField(
        "target_table",
        StringType(),
        False,
    ),
    StructField(
        "target_line_table",
        StringType(),
        True,
    ),
    StructField(
        "changed_records_received",
        LongType(),
        False,
    ),
    StructField(
        "headers_upserted",
        LongType(),
        False,
    ),
    StructField(
        "headers_deleted",
        LongType(),
        False,
    ),
    StructField(
        "lines_replaced",
        LongType(),
        False,
    ),
    StructField(
        "started_utc",
        TimestampType(),
        False,
    ),
    StructField(
        "completed_utc",
        TimestampType(),
        False,
    ),
    StructField(
        "status",
        StringType(),
        False,
    ),
    StructField(
        "error_message",
        StringType(),
        True,
    ),
])


BRONZE_MERGE_RESULTS_DF = spark.createDataFrame(
    bronze_merge_results,
    schema=bronze_merge_schema,
)


# =============================================================================
# 8. VALIDATE THE MERGE RESULTS
# =============================================================================

merge_entity_count = (
    BRONZE_MERGE_RESULTS_DF.count()
)

expected_merge_entity_count = len(
    INCREMENTAL_ENTITIES
)


if (
    merge_entity_count
    != expected_merge_entity_count
):
    raise RuntimeError(
        "Bronze merge-result count does not match the metadata count. "
        f"Expected {expected_merge_entity_count}; "
        f"created {merge_entity_count}."
    )


duplicate_merge_result_count = (
    BRONZE_MERGE_RESULTS_DF
    .groupBy(
        "pipeline_run_id",
        "ingestion_id",
    )
    .count()
    .filter(
        F.col(
            "count"
        ) > 1
    )
    .count()
)


if duplicate_merge_result_count > 0:
    raise RuntimeError(
        "The Bronze merge results contain duplicate ingestion IDs."
    )


successful_merge_count = (
    BRONZE_MERGE_RESULTS_DF
    .filter(
        F.col(
            "status"
        ) == "SUCCEEDED"
    )
    .count()
)


failed_merge_count = (
    BRONZE_MERGE_RESULTS_DF
    .filter(
        F.col(
            "status"
        ) == "FAILED"
    )
    .count()
)


if (
    successful_merge_count
    + failed_merge_count
    != merge_entity_count
):
    raise RuntimeError(
        "Bronze merge status reconciliation failed."
    )


# =============================================================================
# 9. PERSIST THE IDEMPOTENT MERGE AUDIT
# =============================================================================

upsert_idempotent_audit_table(
    source_df=BRONZE_MERGE_RESULTS_DF,
    table_name=BRONZE_MERGE_AUDIT_TABLE,
    merge_condition="""
        target.pipeline_run_id = source.pipeline_run_id
        AND target.ingestion_id = source.ingestion_id
    """,
)


# =============================================================================
# 10. CALCULATE SUMMARY METRICS
# =============================================================================

change_log_row_count = (
    CDC_CHANGE_LOG_BATCH_DF.count()
)


aggregate_merge_metrics = (
    BRONZE_MERGE_RESULTS_DF
    .agg(
        F.coalesce(
            F.sum(
                "changed_records_received"
            ),
            F.lit(
                0
            ),
        ).alias(
            "changed_records_received"
        ),
        F.coalesce(
            F.sum(
                "headers_upserted"
            ),
            F.lit(
                0
            ),
        ).alias(
            "headers_upserted"
        ),
        F.coalesce(
            F.sum(
                "headers_deleted"
            ),
            F.lit(
                0
            ),
        ).alias(
            "headers_deleted"
        ),
        F.coalesce(
            F.sum(
                "lines_replaced"
            ),
            F.lit(
                0
            ),
        ).alias(
            "lines_replaced"
        ),
    )
    .first()
)


TOTAL_CHANGED_RECORDS_RECEIVED = int(
    aggregate_merge_metrics[
        "changed_records_received"
    ]
)

TOTAL_HEADERS_UPSERTED = int(
    aggregate_merge_metrics[
        "headers_upserted"
    ]
)

TOTAL_HEADERS_DELETED = int(
    aggregate_merge_metrics[
        "headers_deleted"
    ]
)

TOTAL_LINES_REPLACED = int(
    aggregate_merge_metrics[
        "lines_replaced"
    ]
)


# =============================================================================
# 11. DISPLAY SAFE MERGE RESULTS
# =============================================================================

display(
    BRONZE_MERGE_RESULTS_DF
    .select(
        "ingestion_id",
        "source_object",
        "target_table",
        "target_line_table",
        "changed_records_received",
        "headers_upserted",
        "headers_deleted",
        "lines_replaced",
        "status",
        "error_message",
    )
    .orderBy(
        "ingestion_id"
    )
)


# =============================================================================
# 12. PUBLISH DOWNSTREAM MERGE STATE
# =============================================================================

BRONZE_MERGE_SUCCEEDED = (
    failed_merge_count == 0
)

BRONZE_MERGE_COMPLETED_WITH_ERRORS = (
    failed_merge_count > 0
)

BRONZE_SUCCESSFUL_ENTITIES = [
    row["source_object"]
    for row in (
        BRONZE_MERGE_RESULTS_DF
        .filter(
            F.col(
                "status"
            ) == "SUCCEEDED"
        )
        .select(
            "source_object"
        )
        .orderBy(
            "source_object"
        )
        .collect()
    )
]

BRONZE_FAILED_ENTITIES = [
    row["source_object"]
    for row in (
        BRONZE_MERGE_RESULTS_DF
        .filter(
            F.col(
                "status"
            ) == "FAILED"
        )
        .select(
            "source_object"
        )
        .orderBy(
            "source_object"
        )
        .collect()
    )
]


# =============================================================================
# 13. SAFE EXECUTION SUMMARY
# =============================================================================

print("=" * 80)
print("QUICKBOOKS INCREMENTAL BRONZE MERGE SUMMARY")
print("=" * 80)
print(f"Pipeline run ID                : {PIPELINE_RUN_ID}")
print(f"Entities expected              : {expected_merge_entity_count}")
print(f"Entities processed             : {merge_entity_count}")
print(f"Entities succeeded             : {successful_merge_count}")
print(f"Entities failed                : {failed_merge_count}")
print(f"CDC change-log batch rows      : {change_log_row_count}")
print(
    "Changed records received      : "
    f"{TOTAL_CHANGED_RECORDS_RECEIVED}"
)
print(f"Headers upserted               : {TOTAL_HEADERS_UPSERTED}")
print(f"Headers deleted                : {TOTAL_HEADERS_DELETED}")
print(f"Current lines written          : {TOTAL_LINES_REPLACED}")
print("=" * 80)


if BRONZE_MERGE_ERRORS:
    if FAIL_ON_ENTITY_ERROR:
        raise RuntimeError(
            "One or more QuickBooks Bronze merges failed:\n"
            + "\n".join(
                f"{number}. {message}"
                for number, message in enumerate(
                    BRONZE_MERGE_ERRORS,
                    start=1,
                )
            )
        )

    print(
        "FAIL_ON_ENTITY_ERROR is False, so successful entities "
        "may continue to downstream validation."
    )
    print(
        "QuickBooks incremental Bronze merge: COMPLETED_WITH_ERRORS"
    )

else:
    print(
        "QuickBooks incremental Bronze merge: SUCCEEDED"
    )

StatementMeta(, 6bf6e71d-4576-4274-981e-6ccc62d6e773, 30, Finished, Available, Finished, False)

Account: 8 headers upserted, 0 headers deleted, 0 previous lines removed, 0 current lines written
Customer: 41 headers upserted, 0 headers deleted, 0 previous lines removed, 0 current lines written
Vendor: 30 headers upserted, 0 headers deleted, 0 previous lines removed, 0 current lines written
Item: 124 headers upserted, 0 headers deleted, 0 previous lines removed, 0 current lines written
Class: 0 headers upserted, 0 headers deleted, 0 previous lines removed, 0 current lines written
Department: 0 headers upserted, 0 headers deleted, 0 previous lines removed, 0 current lines written
Term: 0 headers upserted, 0 headers deleted, 0 previous lines removed, 0 current lines written
Invoice: 0 headers upserted, 0 headers deleted, 0 previous lines removed, 0 current lines written
Payment: 0 headers upserted, 0 headers deleted, 0 previous lines removed, 0 current lines written
CreditMemo: 0 headers upserted, 0 headers deleted, 0 previous lines removed, 0 current lines written
Bill: 2 headers up

SynapseWidget(Synapse.DataFrame, 2fd124fe-2753-4114-aa45-8ace42a42ab6)

QUICKBOOKS INCREMENTAL BRONZE MERGE SUMMARY
Pipeline run ID                : f7ac3015-a3ce-4bf1-bd04-d5d379e38534
Entities expected              : 15
Entities processed             : 15
Entities succeeded             : 15
Entities failed                : 0
CDC change-log batch rows      : 208
Changed records received      : 208
Headers upserted               : 208
Headers deleted                : 0
Current lines written          : 16
QuickBooks incremental Bronze merge: SUCCEEDED


In [29]:
# =============================================================================
# CELL 8 — POST-MERGE BRONZE VALIDATION
# QuickBooks Online Incremental Bronze Ingestion
#
# Purpose:
# 1. Confirm CDC extraction and Bronze MERGE completed successfully.
# 2. Validate mandatory columns, keys and payload retention.
# 3. Detect duplicate current-state business keys.
# 4. Validate header-to-line referential integrity.
# 5. Confirm active CDC records exist in Bronze.
# 6. Confirm deleted records are absent from current-state Bronze.
# 7. Persist idempotent validation audit results.
# 8. Block watermark advancement when any validation check fails.
#
# Prerequisite:
# Cell 7 must publish:
# - BRONZE_MERGE_RESULTS_DF
# - CDC_CHANGE_LOG_BATCH_DF
# - NORMALISED_CDC_RECORDS
# =============================================================================


# -----------------------------------------------------------------------------
# Imports
# -----------------------------------------------------------------------------

from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType,
    StructField,
    StructType,
    TimestampType,
)


# -----------------------------------------------------------------------------
# Constants
# -----------------------------------------------------------------------------

VALIDATION_AUDIT_TABLE = (
    "qbo_es_incremental_validation_audit"
)

VALIDATION_PASSED_STATUS = "PASSED"
VALIDATION_FAILED_STATUS = "FAILED"


# =============================================================================
# 1. VALIDATE REQUIRED UPSTREAM OBJECTS
# =============================================================================

required_objects = [
    "PIPELINE_RUN_ID",
    "INCREMENTAL_ENTITIES",
    "CDC_EXECUTION_RESULTS_DF",
    "BRONZE_MERGE_RESULTS_DF",
    "CDC_CHANGE_LOG_BATCH_DF",
    "NORMALISED_CDC_RECORDS",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Cell 8 cannot run because these upstream objects are missing: "
        + ", ".join(missing_objects)
        + ". Run Cells 1 through 7 first."
    )


if not isinstance(
    INCREMENTAL_ENTITIES,
    list,
):
    raise TypeError(
        "INCREMENTAL_ENTITIES must be a Python list."
    )


if not INCREMENTAL_ENTITIES:
    raise RuntimeError(
        "INCREMENTAL_ENTITIES is empty."
    )


if not isinstance(
    NORMALISED_CDC_RECORDS,
    dict,
):
    raise TypeError(
        "NORMALISED_CDC_RECORDS must be a Python dictionary."
    )


if not str(
    PIPELINE_RUN_ID
).strip():
    raise ValueError(
        "PIPELINE_RUN_ID cannot be blank."
    )


VALIDATION_STARTED_UTC = datetime.now(
    timezone.utc
)


# =============================================================================
# 2. HELPER FUNCTIONS
# =============================================================================

def cell8_utc_now() -> datetime:
    """Return the current timezone-aware UTC timestamp."""
    return datetime.now(
        timezone.utc
    )


def spark_compatible_utc(
    value: Optional[datetime],
) -> Optional[datetime]:
    """
    Convert a datetime into timezone-naive UTC for Spark TimestampType.
    """
    if value is None:
        return None

    if not isinstance(
        value,
        datetime,
    ):
        raise TypeError(
            "Spark timestamp values must be datetime objects or None."
        )

    if value.tzinfo is None:
        return value

    return (
        value
        .astimezone(timezone.utc)
        .replace(tzinfo=None)
    )


def validation_value_to_string(
    value: Any,
) -> Optional[str]:
    """Convert a validation value into a safe string."""
    if value is None:
        return None

    return str(
        value
    )


def get_record_status(
    record: Dict[str, Any],
) -> str:
    """Read either the QuickBooks status or Status field."""
    return str(
        record.get(
            "status",
            record.get(
                "Status",
                "",
            ),
        )
        or ""
    ).strip().lower()


def get_record_id(
    record: Dict[str, Any],
) -> Optional[str]:
    """Return a trimmed QuickBooks record ID or None."""
    record_id = record.get(
        "Id"
    )

    if record_id is None:
        return None

    record_id_text = str(
        record_id
    ).strip()

    return (
        record_id_text
        if record_id_text
        else None
    )


validation_results: List[
    Dict[str, Any]
] = []

validation_errors: List[
    str
] = []


def add_validation_result(
    check_name: str,
    object_name: str,
    observed_value: Any,
    expected_value: Any,
    passed: bool,
    details: Optional[str] = None,
) -> None:
    """Add one validation result and capture failures for the final gate."""
    status = (
        VALIDATION_PASSED_STATUS
        if passed
        else VALIDATION_FAILED_STATUS
    )

    validation_results.append(
        {
            "pipeline_run_id": str(
                PIPELINE_RUN_ID
            ),
            "check_name": str(
                check_name
            ),
            "object_name": str(
                object_name
            ),
            "status": status,
            "observed_value": (
                validation_value_to_string(
                    observed_value
                )
            ),
            "expected_value": (
                validation_value_to_string(
                    expected_value
                )
            ),
            "details": (
                str(details)
                if details is not None
                else None
            ),
            "validated_utc": (
                spark_compatible_utc(
                    cell8_utc_now()
                )
            ),
        }
    )

    if not passed:
        validation_errors.append(
            f"{check_name} failed for {object_name}. "
            f"Observed={observed_value}; "
            f"Expected={expected_value}."
            + (
                f" Details={details}"
                if details
                else ""
            )
        )


def persist_validation_audit(
    validation_df,
) -> None:
    """
    Persist validation results idempotently.

    Rerunning the same pipeline run updates the same validation check
    instead of appending duplicate audit rows.
    """
    if spark.catalog.tableExists(
        VALIDATION_AUDIT_TABLE
    ):
        validation_delta = DeltaTable.forName(
            spark,
            VALIDATION_AUDIT_TABLE,
        )

        (
            validation_delta
            .alias(
                "target"
            )
            .merge(
                validation_df.alias(
                    "source"
                ),
                """
                target.pipeline_run_id = source.pipeline_run_id
                AND target.check_name = source.check_name
                AND target.object_name = source.object_name
                """,
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

    else:
        (
            validation_df
            .write
            .format(
                "delta"
            )
            .mode(
                "overwrite"
            )
            .saveAsTable(
                VALIDATION_AUDIT_TABLE
            )
        )


# =============================================================================
# 3. VALIDATE CDC EXTRACTION AND BRONZE MERGE STATUSES
# =============================================================================

expected_entity_count = len(
    INCREMENTAL_ENTITIES
)


cdc_request_count = (
    CDC_EXECUTION_RESULTS_DF.count()
)

cdc_failed_count = (
    CDC_EXECUTION_RESULTS_DF
    .filter(
        F.col(
            "status"
        ) != "SUCCEEDED"
    )
    .count()
)


merge_result_count = (
    BRONZE_MERGE_RESULTS_DF.count()
)

merge_failed_count = (
    BRONZE_MERGE_RESULTS_DF
    .filter(
        F.col(
            "status"
        ) != "SUCCEEDED"
    )
    .count()
)


add_validation_result(
    check_name="CDC_ENTITY_COUNT",
    object_name="CDC_EXECUTION_RESULTS_DF",
    observed_value=cdc_request_count,
    expected_value=expected_entity_count,
    passed=(
        cdc_request_count
        == expected_entity_count
    ),
)


add_validation_result(
    check_name="CDC_FAILURE_COUNT",
    object_name="CDC_EXECUTION_RESULTS_DF",
    observed_value=cdc_failed_count,
    expected_value=0,
    passed=(
        cdc_failed_count
        == 0
    ),
)


add_validation_result(
    check_name="MERGE_ENTITY_COUNT",
    object_name="BRONZE_MERGE_RESULTS_DF",
    observed_value=merge_result_count,
    expected_value=expected_entity_count,
    passed=(
        merge_result_count
        == expected_entity_count
    ),
)


add_validation_result(
    check_name="MERGE_FAILURE_COUNT",
    object_name="BRONZE_MERGE_RESULTS_DF",
    observed_value=merge_failed_count,
    expected_value=0,
    passed=(
        merge_failed_count
        == 0
    ),
)


# =============================================================================
# 4. RECONCILE CDC RECORD COUNTS
# =============================================================================

total_cdc_changed_records_row = (
    CDC_EXECUTION_RESULTS_DF
    .agg(
        F.coalesce(
            F.sum(
                "changed_records"
            ),
            F.lit(
                0
            ),
        ).alias(
            "total_changed_records"
        )
    )
    .first()
)


total_cdc_changed_records = int(
    total_cdc_changed_records_row[
        "total_changed_records"
    ]
)


change_log_batch_count = (
    CDC_CHANGE_LOG_BATCH_DF.count()
)


add_validation_result(
    check_name="CDC_CHANGE_LOG_RECONCILIATION",
    object_name="qbo_es_cdc_change_log",
    observed_value=change_log_batch_count,
    expected_value=total_cdc_changed_records,
    passed=(
        change_log_batch_count
        == total_cdc_changed_records
    ),
    details=(
        "The current CDC change-log batch must equal the number "
        "of changed records returned by the API."
    ),
)


# =============================================================================
# 5. VALIDATE CURRENT-STATE BRONZE TABLES
# =============================================================================

for entity in INCREMENTAL_ENTITIES:
    source_object = str(
        entity["source_object"]
    ).strip()

    target_table = str(
        entity["target_table"]
    ).strip()

    target_line_table = entity.get(
        "target_line_table"
    )

    if target_line_table is not None:
        target_line_table = str(
            target_line_table
        ).strip() or None


    # -------------------------------------------------------------------------
    # Confirm header table exists
    # -------------------------------------------------------------------------

    header_table_exists = (
        spark.catalog.tableExists(
            target_table
        )
    )

    add_validation_result(
        check_name="HEADER_TABLE_EXISTS",
        object_name=target_table,
        observed_value=header_table_exists,
        expected_value=True,
        passed=header_table_exists,
    )

    if not header_table_exists:
        continue


    header_df = spark.table(
        target_table
    )


    # -------------------------------------------------------------------------
    # Validate required header columns
    # -------------------------------------------------------------------------

    required_header_columns = {
        "source_record_id",
        "payload_json",
        "payload_hash",
        "_source_system",
        "_source_company",
        "_source_entity",
        "_pipeline_run_id",
        "_ingested_utc",
    }

    missing_header_columns = sorted(
        required_header_columns
        - set(
            header_df.columns
        )
    )

    add_validation_result(
        check_name="HEADER_REQUIRED_COLUMNS",
        object_name=target_table,
        observed_value=(
            "All present"
            if not missing_header_columns
            else ", ".join(
                missing_header_columns
            )
        ),
        expected_value="All required columns",
        passed=(
            not missing_header_columns
        ),
    )

    if missing_header_columns:
        continue


    # -------------------------------------------------------------------------
    # Mandatory source key
    # -------------------------------------------------------------------------

    invalid_header_id_count = (
        header_df
        .filter(
            F.col(
                "source_record_id"
            ).isNull()
            | (
                F.trim(
                    F.col(
                        "source_record_id"
                    )
                )
                == ""
            )
        )
        .count()
    )

    add_validation_result(
        check_name="MANDATORY_SOURCE_RECORD_ID",
        object_name=target_table,
        observed_value=invalid_header_id_count,
        expected_value=0,
        passed=(
            invalid_header_id_count
            == 0
        ),
    )


    # -------------------------------------------------------------------------
    # Current-state business-key uniqueness
    # -------------------------------------------------------------------------

    duplicate_header_key_count = (
        header_df
        .groupBy(
            "source_record_id"
        )
        .count()
        .filter(
            F.col(
                "count"
            ) > 1
        )
        .count()
    )

    add_validation_result(
        check_name="DUPLICATE_SOURCE_RECORD_ID",
        object_name=target_table,
        observed_value=duplicate_header_key_count,
        expected_value=0,
        passed=(
            duplicate_header_key_count
            == 0
        ),
    )


    # -------------------------------------------------------------------------
    # Payload retention
    # -------------------------------------------------------------------------

    missing_payload_count = (
        header_df
        .filter(
            F.col(
                "payload_json"
            ).isNull()
            | (
                F.trim(
                    F.col(
                        "payload_json"
                    )
                )
                == ""
            )
        )
        .count()
    )

    add_validation_result(
        check_name="PAYLOAD_PRESENT",
        object_name=target_table,
        observed_value=missing_payload_count,
        expected_value=0,
        passed=(
            missing_payload_count
            == 0
        ),
    )


    missing_payload_hash_count = (
        header_df
        .filter(
            F.col(
                "payload_hash"
            ).isNull()
            | (
                F.trim(
                    F.col(
                        "payload_hash"
                    )
                )
                == ""
            )
        )
        .count()
    )

    add_validation_result(
        check_name="PAYLOAD_HASH_PRESENT",
        object_name=target_table,
        observed_value=missing_payload_hash_count,
        expected_value=0,
        passed=(
            missing_payload_hash_count
            == 0
        ),
    )


    # -------------------------------------------------------------------------
    # Validate changed and deleted CDC records
    # -------------------------------------------------------------------------

    entity_records = (
        NORMALISED_CDC_RECORDS.get(
            source_object,
            [],
        )
    )

    active_changed_ids = sorted({
        record_id
        for record in entity_records
        for record_id in [
            get_record_id(
                record
            )
        ]
        if (
            record_id is not None
            and get_record_status(
                record
            ) != "deleted"
        )
    })


    deleted_ids = sorted({
        record_id
        for record in entity_records
        for record_id in [
            get_record_id(
                record
            )
        ]
        if (
            record_id is not None
            and get_record_status(
                record
            ) == "deleted"
        )
    })


    if active_changed_ids:
        active_ids_found_count = (
            header_df
            .filter(
                F.col(
                    "source_record_id"
                ).isin(
                    active_changed_ids
                )
            )
            .select(
                "source_record_id"
            )
            .distinct()
            .count()
        )
    else:
        active_ids_found_count = 0


    add_validation_result(
        check_name="ACTIVE_CHANGED_IDS_PRESENT",
        object_name=target_table,
        observed_value=active_ids_found_count,
        expected_value=len(
            active_changed_ids
        ),
        passed=(
            active_ids_found_count
            == len(
                active_changed_ids
            )
        ),
    )


    if deleted_ids:
        deleted_ids_still_present = (
            header_df
            .filter(
                F.col(
                    "source_record_id"
                ).isin(
                    deleted_ids
                )
            )
            .count()
        )
    else:
        deleted_ids_still_present = 0


    add_validation_result(
        check_name="DELETED_IDS_ABSENT",
        object_name=target_table,
        observed_value=deleted_ids_still_present,
        expected_value=0,
        passed=(
            deleted_ids_still_present
            == 0
        ),
    )


    # -------------------------------------------------------------------------
    # Validate source metadata values in the current-state table
    # -------------------------------------------------------------------------

    incorrect_source_system_count = (
        header_df
        .filter(
            F.col(
                "_source_system"
            ) != str(
                entity["source_system"]
            )
        )
        .count()
    )

    add_validation_result(
        check_name="SOURCE_SYSTEM_CONSISTENCY",
        object_name=target_table,
        observed_value=incorrect_source_system_count,
        expected_value=0,
        passed=(
            incorrect_source_system_count
            == 0
        ),
    )


    incorrect_source_company_count = (
        header_df
        .filter(
            F.col(
                "_source_company"
            ) != str(
                entity["source_company"]
            )
        )
        .count()
    )

    add_validation_result(
        check_name="SOURCE_COMPANY_CONSISTENCY",
        object_name=target_table,
        observed_value=incorrect_source_company_count,
        expected_value=0,
        passed=(
            incorrect_source_company_count
            == 0
        ),
    )


    incorrect_source_entity_count = (
        header_df
        .filter(
            F.col(
                "_source_entity"
            ) != source_object
        )
        .count()
    )

    add_validation_result(
        check_name="SOURCE_ENTITY_CONSISTENCY",
        object_name=target_table,
        observed_value=incorrect_source_entity_count,
        expected_value=0,
        passed=(
            incorrect_source_entity_count
            == 0
        ),
    )


    # -------------------------------------------------------------------------
    # Validate line table when configured
    # -------------------------------------------------------------------------

    if not target_line_table:
        continue


    line_table_exists = (
        spark.catalog.tableExists(
            target_line_table
        )
    )

    add_validation_result(
        check_name="LINE_TABLE_EXISTS",
        object_name=target_line_table,
        observed_value=line_table_exists,
        expected_value=True,
        passed=line_table_exists,
    )

    if not line_table_exists:
        continue


    line_df = spark.table(
        target_line_table
    )


    required_line_columns = {
        "parent_record_id",
        "line_number",
        "line_payload_json",
        "line_payload_hash",
        "_source_system",
        "_source_company",
        "_source_entity",
        "_pipeline_run_id",
        "_ingested_utc",
    }

    missing_line_columns = sorted(
        required_line_columns
        - set(
            line_df.columns
        )
    )

    add_validation_result(
        check_name="LINE_REQUIRED_COLUMNS",
        object_name=target_line_table,
        observed_value=(
            "All present"
            if not missing_line_columns
            else ", ".join(
                missing_line_columns
            )
        ),
        expected_value="All required columns",
        passed=(
            not missing_line_columns
        ),
    )

    if missing_line_columns:
        continue


    # -------------------------------------------------------------------------
    # Mandatory parent key
    # -------------------------------------------------------------------------

    invalid_parent_id_count = (
        line_df
        .filter(
            F.col(
                "parent_record_id"
            ).isNull()
            | (
                F.trim(
                    F.col(
                        "parent_record_id"
                    )
                )
                == ""
            )
        )
        .count()
    )

    add_validation_result(
        check_name="MANDATORY_PARENT_RECORD_ID",
        object_name=target_line_table,
        observed_value=invalid_parent_id_count,
        expected_value=0,
        passed=(
            invalid_parent_id_count
            == 0
        ),
    )


    # -------------------------------------------------------------------------
    # Mandatory line number
    # -------------------------------------------------------------------------

    invalid_line_number_count = (
        line_df
        .filter(
            F.col(
                "line_number"
            ).isNull()
            | (
                F.col(
                    "line_number"
                ) <= 0
            )
        )
        .count()
    )

    add_validation_result(
        check_name="MANDATORY_LINE_NUMBER",
        object_name=target_line_table,
        observed_value=invalid_line_number_count,
        expected_value=0,
        passed=(
            invalid_line_number_count
            == 0
        ),
    )


    # -------------------------------------------------------------------------
    # Line-key uniqueness
    # -------------------------------------------------------------------------

    duplicate_line_key_count = (
        line_df
        .groupBy(
            "parent_record_id",
            "line_number",
        )
        .count()
        .filter(
            F.col(
                "count"
            ) > 1
        )
        .count()
    )

    add_validation_result(
        check_name="DUPLICATE_LINE_KEY",
        object_name=target_line_table,
        observed_value=duplicate_line_key_count,
        expected_value=0,
        passed=(
            duplicate_line_key_count
            == 0
        ),
    )


    # -------------------------------------------------------------------------
    # Line payload retention
    # -------------------------------------------------------------------------

    missing_line_payload_count = (
        line_df
        .filter(
            F.col(
                "line_payload_json"
            ).isNull()
            | (
                F.trim(
                    F.col(
                        "line_payload_json"
                    )
                )
                == ""
            )
        )
        .count()
    )

    add_validation_result(
        check_name="LINE_PAYLOAD_PRESENT",
        object_name=target_line_table,
        observed_value=missing_line_payload_count,
        expected_value=0,
        passed=(
            missing_line_payload_count
            == 0
        ),
    )


    missing_line_payload_hash_count = (
        line_df
        .filter(
            F.col(
                "line_payload_hash"
            ).isNull()
            | (
                F.trim(
                    F.col(
                        "line_payload_hash"
                    )
                )
                == ""
            )
        )
        .count()
    )

    add_validation_result(
        check_name="LINE_PAYLOAD_HASH_PRESENT",
        object_name=target_line_table,
        observed_value=missing_line_payload_hash_count,
        expected_value=0,
        passed=(
            missing_line_payload_hash_count
            == 0
        ),
    )


    # -------------------------------------------------------------------------
    # Header-to-line referential integrity
    # -------------------------------------------------------------------------

    header_keys_df = (
        header_df
        .select(
            F.col(
                "source_record_id"
            ).alias(
                "parent_record_id"
            )
        )
        .dropDuplicates()
    )


    orphan_line_count = (
        line_df
        .join(
            header_keys_df,
            on="parent_record_id",
            how="left_anti",
        )
        .count()
    )

    add_validation_result(
        check_name="HEADER_LINE_INTEGRITY",
        object_name=target_line_table,
        observed_value=orphan_line_count,
        expected_value=0,
        passed=(
            orphan_line_count
            == 0
        ),
        details=(
            f"Parent table: {target_table}"
        ),
    )


    # -------------------------------------------------------------------------
    # Confirm deleted parent lines are absent
    # -------------------------------------------------------------------------

    if deleted_ids:
        deleted_parent_line_count = (
            line_df
            .filter(
                F.col(
                    "parent_record_id"
                ).isin(
                    deleted_ids
                )
            )
            .count()
        )
    else:
        deleted_parent_line_count = 0


    add_validation_result(
        check_name="DELETED_PARENT_LINES_ABSENT",
        object_name=target_line_table,
        observed_value=deleted_parent_line_count,
        expected_value=0,
        passed=(
            deleted_parent_line_count
            == 0
        ),
    )


# =============================================================================
# 6. BUILD THE VALIDATION DATAFRAME
# =============================================================================

if not validation_results:
    raise RuntimeError(
        "No post-merge validation checks were generated."
    )


validation_schema = StructType([
    StructField(
        "pipeline_run_id",
        StringType(),
        False,
    ),
    StructField(
        "check_name",
        StringType(),
        False,
    ),
    StructField(
        "object_name",
        StringType(),
        False,
    ),
    StructField(
        "status",
        StringType(),
        False,
    ),
    StructField(
        "observed_value",
        StringType(),
        True,
    ),
    StructField(
        "expected_value",
        StringType(),
        True,
    ),
    StructField(
        "details",
        StringType(),
        True,
    ),
    StructField(
        "validated_utc",
        TimestampType(),
        False,
    ),
])


INCREMENTAL_VALIDATION_DF = spark.createDataFrame(
    validation_results,
    schema=validation_schema,
)


# =============================================================================
# 7. VALIDATE THE VALIDATION DATAFRAME
# =============================================================================

total_check_count = (
    INCREMENTAL_VALIDATION_DF.count()
)


duplicate_validation_check_count = (
    INCREMENTAL_VALIDATION_DF
    .groupBy(
        "pipeline_run_id",
        "check_name",
        "object_name",
    )
    .count()
    .filter(
        F.col(
            "count"
        ) > 1
    )
    .count()
)


if duplicate_validation_check_count > 0:
    raise RuntimeError(
        "Duplicate validation checks were generated for the same "
        "pipeline run, check name and object."
    )


invalid_validation_status_count = (
    INCREMENTAL_VALIDATION_DF
    .filter(
        ~F.col(
            "status"
        ).isin(
            VALIDATION_PASSED_STATUS,
            VALIDATION_FAILED_STATUS,
        )
    )
    .count()
)


if invalid_validation_status_count > 0:
    raise RuntimeError(
        "The validation DataFrame contains invalid status values."
    )


passed_check_count = (
    INCREMENTAL_VALIDATION_DF
    .filter(
        F.col(
            "status"
        ) == VALIDATION_PASSED_STATUS
    )
    .count()
)


failed_check_count = (
    INCREMENTAL_VALIDATION_DF
    .filter(
        F.col(
            "status"
        ) == VALIDATION_FAILED_STATUS
    )
    .count()
)


if (
    passed_check_count
    + failed_check_count
    != total_check_count
):
    raise RuntimeError(
        "Validation status reconciliation failed."
    )


# =============================================================================
# 8. PERSIST THE IDEMPOTENT VALIDATION AUDIT
# =============================================================================

persist_validation_audit(
    INCREMENTAL_VALIDATION_DF
)


# =============================================================================
# 9. DISPLAY SAFE VALIDATION RESULTS
# =============================================================================

display(
    INCREMENTAL_VALIDATION_DF
    .orderBy(
        F.col(
            "status"
        ).asc(),
        F.col(
            "object_name"
        ).asc(),
        F.col(
            "check_name"
        ).asc(),
    )
)


# =============================================================================
# 10. PUBLISH VALIDATION STATE
# =============================================================================

POST_MERGE_VALIDATION_SUCCEEDED = (
    failed_check_count == 0
)

POST_MERGE_VALIDATION_FAILED = (
    failed_check_count > 0
)

POST_MERGE_VALIDATION_CHECK_COUNT = (
    total_check_count
)

POST_MERGE_VALIDATION_FAILURE_COUNT = (
    failed_check_count
)


# =============================================================================
# 11. SAFE VALIDATION SUMMARY AND FINAL GATE
# =============================================================================

print("=" * 80)
print("QUICKBOOKS POST-MERGE VALIDATION SUMMARY")
print("=" * 80)
print(f"Pipeline run ID : {PIPELINE_RUN_ID}")
print(f"Checks executed : {total_check_count}")
print(f"Checks passed   : {passed_check_count}")
print(f"Checks failed   : {failed_check_count}")
print("=" * 80)


if validation_errors:
    complete_error_message = (
        "QuickBooks post-merge validation failed:\n"
        + "\n".join(
            f"{number}. {message}"
            for number, message in enumerate(
                validation_errors,
                start=1,
            )
        )
    )

    print(
        complete_error_message
    )

    raise RuntimeError(
        complete_error_message
    )


print(
    "QuickBooks post-merge validation: SUCCEEDED"
)

StatementMeta(, 6bf6e71d-4576-4274-981e-6ccc62d6e773, 31, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e6dc0c9b-a1dc-4c53-9362-bf36f9d0e7a7)

QUICKBOOKS POST-MERGE VALIDATION SUMMARY
Pipeline run ID : f7ac3015-a3ce-4bf1-bd04-d5d379e38534
Checks executed : 233
Checks passed   : 233
Checks failed   : 0
QuickBooks post-merge validation: SUCCEEDED


In [30]:
# =============================================================================
# CELL 9 — STAGE SUCCESSFUL WATERMARK UPDATES
# QuickBooks Online Incremental Bronze Ingestion
#
# Purpose:
# 1. Run only after CDC extraction, Bronze MERGE and validation succeed.
# 2. Prepare one successful watermark row per processed QBO entity.
# 3. Use the fixed extraction upper boundary created by Cell 5A.
# 4. Persist an idempotent Delta staging table.
# 5. Allow the pipeline Script activity to update ctl.watermark_tracker.
#
# Important:
# This cell does not update the Control Warehouse directly.
# The child-pipeline Script activity performs the final Warehouse MERGE.
#
# Watermark safety:
# Watermarks must advance only to EXTRACTION_END_UTC—the upper boundary
# actually used for the successful CDC extraction. This cell must never
# invent a replacement timestamp.
# =============================================================================


# -----------------------------------------------------------------------------
# Imports
# -----------------------------------------------------------------------------

from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType,
    StructField,
    StructType,
    TimestampType,
)


# -----------------------------------------------------------------------------
# Constants
# -----------------------------------------------------------------------------

WATERMARK_STAGE_TABLE = (
    "qbo_es_watermark_update_stage"
)

SUCCESSFUL_RUN_STATUS = "SUCCEEDED"


# =============================================================================
# 1. VALIDATE REQUIRED UPSTREAM OBJECTS
# =============================================================================

required_objects = [
    "PIPELINE_RUN_ID",
    "INCREMENTAL_ENTITIES",
    "EXTRACTION_END_UTC",
    "CDC_EXTRACTION_SUCCEEDED",
    "BRONZE_MERGE_SUCCEEDED",
    "POST_MERGE_VALIDATION_SUCCEEDED",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Cell 9 cannot run because these upstream objects are missing: "
        + ", ".join(missing_objects)
        + ". Run Cells 1 through 8 first."
    )


if not str(
    PIPELINE_RUN_ID
).strip():
    raise ValueError(
        "PIPELINE_RUN_ID cannot be blank."
    )


if not isinstance(
    INCREMENTAL_ENTITIES,
    list,
):
    raise TypeError(
        "INCREMENTAL_ENTITIES must be a Python list."
    )


if not INCREMENTAL_ENTITIES:
    raise RuntimeError(
        "INCREMENTAL_ENTITIES is empty."
    )


# =============================================================================
# 2. ENFORCE THE SUCCESS GATES
# =============================================================================

if CDC_EXTRACTION_SUCCEEDED is not True:
    raise RuntimeError(
        "Watermarks cannot advance because CDC extraction "
        "did not complete successfully."
    )


if BRONZE_MERGE_SUCCEEDED is not True:
    raise RuntimeError(
        "Watermarks cannot advance because the Bronze MERGE "
        "did not complete successfully."
    )


if POST_MERGE_VALIDATION_SUCCEEDED is not True:
    raise RuntimeError(
        "Watermarks cannot advance because post-merge validation "
        "did not complete successfully."
    )


# Optional defensive checks against failure collections.
if (
    "CDC_FAILED_ENTITIES" in globals()
    and CDC_FAILED_ENTITIES
):
    raise RuntimeError(
        "Watermarks cannot advance because CDC failures exist for: "
        + ", ".join(
            sorted(
                str(value)
                for value in CDC_FAILED_ENTITIES
            )
        )
    )


if (
    "BRONZE_FAILED_ENTITIES" in globals()
    and BRONZE_FAILED_ENTITIES
):
    raise RuntimeError(
        "Watermarks cannot advance because Bronze MERGE failures "
        "exist for: "
        + ", ".join(
            sorted(
                str(value)
                for value in BRONZE_FAILED_ENTITIES
            )
        )
    )


# =============================================================================
# 3. RESOLVE THE SUCCESSFUL WATERMARK TIMESTAMP
# =============================================================================

def normalise_utc_datetime(
    value: Any,
    field_name: str,
) -> datetime:
    """
    Convert a supported datetime value into a timezone-aware UTC datetime.

    A missing value raises an error because a watermark must never be
    advanced using an invented timestamp.
    """
    if value is None:
        raise ValueError(
            f"{field_name} cannot be null."
        )

    if isinstance(
        value,
        datetime,
    ):
        parsed_value = value

    else:
        value_text = str(
            value
        ).strip()

        if not value_text:
            raise ValueError(
                f"{field_name} cannot be blank."
            )

        normalised_text = value_text.replace(
            "Z",
            "+00:00",
        )

        try:
            parsed_value = datetime.fromisoformat(
                normalised_text
            )

        except ValueError as exc:
            raise ValueError(
                f"{field_name} is not a valid datetime: {value!r}"
            ) from exc

    if parsed_value.tzinfo is None:
        parsed_value = parsed_value.replace(
            tzinfo=timezone.utc
        )

    return parsed_value.astimezone(
        timezone.utc
    )


def spark_compatible_utc(
    value: datetime,
) -> datetime:
    """
    Convert a timezone-aware datetime into timezone-naive UTC for
    Spark TimestampType.
    """
    value_utc = normalise_utc_datetime(
        value=value,
        field_name="Spark timestamp",
    )

    return value_utc.replace(
        tzinfo=None
    )


SUCCESSFUL_WATERMARK_UTC_AWARE = (
    normalise_utc_datetime(
        value=EXTRACTION_END_UTC,
        field_name="EXTRACTION_END_UTC",
    )
)

SUCCESSFUL_WATERMARK_UTC = (
    spark_compatible_utc(
        SUCCESSFUL_WATERMARK_UTC_AWARE
    )
)

PREPARED_UTC_AWARE = datetime.now(
    timezone.utc
)

PREPARED_UTC = spark_compatible_utc(
    PREPARED_UTC_AWARE
)


if (
    SUCCESSFUL_WATERMARK_UTC_AWARE
    > PREPARED_UTC_AWARE
):
    raise RuntimeError(
        "EXTRACTION_END_UTC cannot be later than the current UTC time."
    )


# =============================================================================
# 4. BUILD ONE WATERMARK ROW PER SUCCESSFUL ENTITY
# =============================================================================

watermark_rows: List[
    Dict[str, Any]
] = []

watermark_preparation_errors: List[
    str
] = []


for entity_position, entity in enumerate(
    INCREMENTAL_ENTITIES,
    start=1,
):
    if not isinstance(
        entity,
        dict,
    ):
        watermark_preparation_errors.append(
            f"Entity {entity_position} is not a Python dictionary."
        )
        continue

    source_system = str(
        entity.get(
            "source_system",
            "",
        )
        or ""
    ).strip().upper()

    source_company = str(
        entity.get(
            "source_company",
            "",
        )
        or ""
    ).strip().upper()

    source_object = str(
        entity.get(
            "source_object",
            "",
        )
        or ""
    ).strip()

    if not source_system:
        watermark_preparation_errors.append(
            f"Entity {entity_position} has a blank source_system."
        )

    if not source_company:
        watermark_preparation_errors.append(
            f"Entity {entity_position} has a blank source_company."
        )

    if not source_object:
        watermark_preparation_errors.append(
            f"Entity {entity_position} has a blank source_object."
        )

    if (
        not source_system
        or not source_company
        or not source_object
    ):
        continue

    watermark_rows.append(
        {
            "pipeline_run_id": str(
                PIPELINE_RUN_ID
            ).strip(),
            "source_system": source_system,
            "source_company": source_company,
            "source_object": source_object,
            "successful_watermark_utc":
                SUCCESSFUL_WATERMARK_UTC,
            "run_status":
                SUCCESSFUL_RUN_STATUS,
            "prepared_utc":
                PREPARED_UTC,
        }
    )


if watermark_preparation_errors:
    raise RuntimeError(
        "Watermark-row preparation failed:\n"
        + "\n".join(
            f"{number}. {message}"
            for number, message in enumerate(
                watermark_preparation_errors,
                start=1,
            )
        )
    )


if not watermark_rows:
    raise RuntimeError(
        "No successful watermark rows were prepared."
    )


# =============================================================================
# 5. DEFINE THE EXPLICIT SPARK SCHEMA
# =============================================================================

watermark_schema = StructType([
    StructField(
        "pipeline_run_id",
        StringType(),
        False,
    ),
    StructField(
        "source_system",
        StringType(),
        False,
    ),
    StructField(
        "source_company",
        StringType(),
        False,
    ),
    StructField(
        "source_object",
        StringType(),
        False,
    ),
    StructField(
        "successful_watermark_utc",
        TimestampType(),
        False,
    ),
    StructField(
        "run_status",
        StringType(),
        False,
    ),
    StructField(
        "prepared_utc",
        TimestampType(),
        False,
    ),
])


WATERMARK_UPDATE_STAGE_DF = spark.createDataFrame(
    watermark_rows,
    schema=watermark_schema,
)


# =============================================================================
# 6. STANDARDISE AND DEDUPLICATE THE STAGING DATASET
# =============================================================================

WATERMARK_UPDATE_STAGE_DF = (
    WATERMARK_UPDATE_STAGE_DF
    .withColumn(
        "pipeline_run_id",
        F.trim(
            F.col(
                "pipeline_run_id"
            )
        ),
    )
    .withColumn(
        "source_system",
        F.upper(
            F.trim(
                F.col(
                    "source_system"
                )
            )
        ),
    )
    .withColumn(
        "source_company",
        F.upper(
            F.trim(
                F.col(
                    "source_company"
                )
            )
        ),
    )
    .withColumn(
        "source_object",
        F.trim(
            F.col(
                "source_object"
            )
        ),
    )
    .withColumn(
        "run_status",
        F.upper(
            F.trim(
                F.col(
                    "run_status"
                )
            )
        ),
    )
    .dropDuplicates([
        "pipeline_run_id",
        "source_system",
        "source_company",
        "source_object",
    ])
)


# =============================================================================
# 7. VALIDATE THE PREPARED STAGING DATA
# =============================================================================

expected_entity_keys = {
    (
        str(
            entity["source_system"]
        ).strip().upper(),
        str(
            entity["source_company"]
        ).strip().upper(),
        str(
            entity["source_object"]
        ).strip(),
    )
    for entity in INCREMENTAL_ENTITIES
}


expected_entity_count = len(
    expected_entity_keys
)

prepared_entity_count = (
    WATERMARK_UPDATE_STAGE_DF.count()
)


if (
    prepared_entity_count
    != expected_entity_count
):
    raise RuntimeError(
        "Watermark staging row-count validation failed. "
        f"Expected {expected_entity_count}; "
        f"prepared {prepared_entity_count}."
    )


null_mandatory_value_count = (
    WATERMARK_UPDATE_STAGE_DF
    .filter(
        F.col(
            "pipeline_run_id"
        ).isNull()
        | F.col(
            "source_system"
        ).isNull()
        | F.col(
            "source_company"
        ).isNull()
        | F.col(
            "source_object"
        ).isNull()
        | F.col(
            "successful_watermark_utc"
        ).isNull()
        | F.col(
            "run_status"
        ).isNull()
        | F.col(
            "prepared_utc"
        ).isNull()
    )
    .count()
)


if null_mandatory_value_count > 0:
    raise RuntimeError(
        "Watermark staging contains null mandatory values. "
        f"Invalid rows: {null_mandatory_value_count}."
    )


blank_mandatory_value_count = (
    WATERMARK_UPDATE_STAGE_DF
    .filter(
        (
            F.trim(
                F.col(
                    "pipeline_run_id"
                )
            )
            == ""
        )
        | (
            F.trim(
                F.col(
                    "source_system"
                )
            )
            == ""
        )
        | (
            F.trim(
                F.col(
                    "source_company"
                )
            )
            == ""
        )
        | (
            F.trim(
                F.col(
                    "source_object"
                )
            )
            == ""
        )
        | (
            F.trim(
                F.col(
                    "run_status"
                )
            )
            == ""
        )
    )
    .count()
)


if blank_mandatory_value_count > 0:
    raise RuntimeError(
        "Watermark staging contains blank mandatory values. "
        f"Invalid rows: {blank_mandatory_value_count}."
    )


invalid_status_count = (
    WATERMARK_UPDATE_STAGE_DF
    .filter(
        F.col(
            "run_status"
        )
        != SUCCESSFUL_RUN_STATUS
    )
    .count()
)


if invalid_status_count > 0:
    raise RuntimeError(
        "Watermark staging contains a status other than SUCCEEDED."
    )


duplicate_entity_count = (
    WATERMARK_UPDATE_STAGE_DF
    .groupBy(
        "pipeline_run_id",
        "source_system",
        "source_company",
        "source_object",
    )
    .count()
    .filter(
        F.col(
            "count"
        ) > 1
    )
    .count()
)


if duplicate_entity_count > 0:
    raise RuntimeError(
        "Duplicate entity watermark rows were prepared."
    )


# =============================================================================
# 8. UPSERT INTO THE BRONZE WATERMARK STAGING TABLE
# =============================================================================

if spark.catalog.tableExists(
    WATERMARK_STAGE_TABLE
):
    existing_stage_columns = set(
        spark.table(
            WATERMARK_STAGE_TABLE
        ).columns
    )

    required_stage_columns = set(
        WATERMARK_UPDATE_STAGE_DF.columns
    )

    missing_stage_columns = sorted(
        required_stage_columns
        - existing_stage_columns
    )

    if missing_stage_columns:
        raise RuntimeError(
            f"{WATERMARK_STAGE_TABLE} is missing required columns: "
            + ", ".join(
                missing_stage_columns
            )
        )

    watermark_stage_delta = DeltaTable.forName(
        spark,
        WATERMARK_STAGE_TABLE,
    )

    (
        watermark_stage_delta
        .alias(
            "target"
        )
        .merge(
            WATERMARK_UPDATE_STAGE_DF.alias(
                "source"
            ),
            """
            target.pipeline_run_id = source.pipeline_run_id
            AND target.source_system = source.source_system
            AND target.source_company = source.source_company
            AND target.source_object = source.source_object
            """,
        )
        .whenMatchedUpdate(
            set={
                "successful_watermark_utc":
                    "source.successful_watermark_utc",
                "run_status":
                    "source.run_status",
                "prepared_utc":
                    "source.prepared_utc",
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

else:
    (
        WATERMARK_UPDATE_STAGE_DF
        .write
        .format(
            "delta"
        )
        .mode(
            "overwrite"
        )
        .saveAsTable(
            WATERMARK_STAGE_TABLE
        )
    )


# =============================================================================
# 9. VERIFY THE CURRENT PIPELINE RUN AFTER PERSISTENCE
# =============================================================================

staged_run_df = (
    spark.table(
        WATERMARK_STAGE_TABLE
    )
    .filter(
        F.col(
            "pipeline_run_id"
        )
        == str(
            PIPELINE_RUN_ID
        ).strip()
    )
)


staged_entity_count = (
    staged_run_df
    .select(
        "source_system",
        "source_company",
        "source_object",
    )
    .dropDuplicates()
    .count()
)


if (
    staged_entity_count
    != expected_entity_count
):
    raise RuntimeError(
        "Post-write watermark staging validation failed. "
        f"Expected {expected_entity_count} entities for run "
        f"{PIPELINE_RUN_ID}; found {staged_entity_count}."
    )


staged_invalid_status_count = (
    staged_run_df
    .filter(
        F.col(
            "run_status"
        )
        != SUCCESSFUL_RUN_STATUS
    )
    .count()
)


if staged_invalid_status_count > 0:
    raise RuntimeError(
        "The persisted watermark stage contains a non-successful status."
    )


staged_watermark_count = (
    staged_run_df
    .select(
        "successful_watermark_utc"
    )
    .distinct()
    .count()
)


if staged_watermark_count != 1:
    raise RuntimeError(
        "The current pipeline run contains more than one successful "
        "watermark timestamp."
    )


persisted_watermark = (
    staged_run_df
    .select(
        "successful_watermark_utc"
    )
    .first()[
        "successful_watermark_utc"
    ]
)


if (
    persisted_watermark
    != SUCCESSFUL_WATERMARK_UTC
):
    raise RuntimeError(
        "The persisted successful watermark does not match "
        "EXTRACTION_END_UTC."
    )


# =============================================================================
# 10. DISPLAY AND PUBLISH DOWNSTREAM STATE
# =============================================================================

display(
    staged_run_df
    .orderBy(
        "source_object"
    )
)


WATERMARK_STAGE_SUCCEEDED = True

WATERMARK_STAGE_ENTITY_COUNT = (
    staged_entity_count
)

WATERMARK_STAGE_TABLE_NAME = (
    WATERMARK_STAGE_TABLE
)

WATERMARK_STAGE_SUCCESSFUL_UTC = (
    SUCCESSFUL_WATERMARK_UTC
)


# =============================================================================
# 11. SAFE EXECUTION SUMMARY
# =============================================================================

print("=" * 80)
print("QBO WATERMARK UPDATE STAGING SUMMARY")
print("=" * 80)
print(f"Pipeline run ID          : {PIPELINE_RUN_ID}")
print(f"Entities expected        : {expected_entity_count}")
print(f"Entities staged          : {staged_entity_count}")
print(
    "Successful watermark UTC: "
    f"{SUCCESSFUL_WATERMARK_UTC_AWARE.isoformat()}"
)
print(f"Run status               : {SUCCESSFUL_RUN_STATUS}")
print(f"Staging table            : {WATERMARK_STAGE_TABLE}")
print("Watermark staging        : SUCCEEDED")
print("=" * 80)

StatementMeta(, 6bf6e71d-4576-4274-981e-6ccc62d6e773, 32, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0a05e05a-5ccb-42ad-8e41-e29195907d4a)

QBO WATERMARK UPDATE STAGING SUMMARY
Pipeline run ID          : f7ac3015-a3ce-4bf1-bd04-d5d379e38534
Entities expected        : 15
Entities staged          : 15
Successful watermark UTC: 2026-08-04T23:56:38.891662+00:00
Run status               : SUCCEEDED
Staging table            : qbo_es_watermark_update_stage
Watermark staging        : SUCCEEDED


In [31]:
display(
    spark.sql("""
        SELECT *
        FROM qbo_es_incremental_validation_audit
        ORDER BY validated_utc DESC
        LIMIT 25
    """)
)

StatementMeta(, 6bf6e71d-4576-4274-981e-6ccc62d6e773, 33, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 02333449-7398-433d-9c9a-1f51557030ff)

In [32]:
# =============================================================================
# CELL 10 - VALIDATION AUDIT REVIEW (READ ONLY)
# Purpose:
# Review the latest validation audit records after a successful pipeline run.
# This cell is for monitoring and troubleshooting only.
# It does not modify any data.
# =============================================================================

validation_df = spark.table(
    "qbo_es_incremental_validation_audit"
)

print("=" * 80)
print("VALIDATION AUDIT SUMMARY")
print("=" * 80)
print(f"Total audit records : {validation_df.count()}")
print("=" * 80)

display(
    validation_df
        .orderBy("validated_utc", ascending=False)
        .limit(50)
)

StatementMeta(, 6bf6e71d-4576-4274-981e-6ccc62d6e773, 34, Finished, Available, Finished, False)

VALIDATION AUDIT SUMMARY
Total audit records : 2421


SynapseWidget(Synapse.DataFrame, ea5f749b-8f6c-4488-a310-e9e74230ce52)